# Aneurysm CFD: the OpenFOAM + VMTK workflow, rebuilt with tensors

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/aneurysm_cfd)

Pulsatile blood flow through a real intracranial aneurysm, segmented from an open (CC0) TOF-MRA scan, following the standard research workflow (VMTK centrelines and flow extensions, an OpenFOAM-style incompressible solver, then wall shear stress, TAWSS and OSI) with every step written as tensor operations. The solver is checked against Poiseuille and Womersley flow and against OpenFOAM runs on a pipe and on the same aneurysm mesh, the centrelines against VMTK. Educational only, not for clinical use.

**Vector form**

$$\partial_t\mathbf u + (\mathbf u\cdot\nabla)\mathbf u = -\tfrac{1}{\rho}\nabla p + \nu\nabla^2\mathbf u,\quad \nabla\cdot\mathbf u = 0,\qquad \boldsymbol\tau_w = \mu\,(\mathbf I - \mathbf n\mathbf n)\,\partial_n\mathbf u,\qquad |\nabla T| = 1/R^2$$

**Index (tensor) form**

$$(\nabla\!\cdot\mathbf u)_c = \tfrac1h\Big(\sum_{o(f)=c}u_f - \sum_{n(f)=c}u_f\Big),\quad A_{cd}\,p_d = -\tfrac{1}{\Delta t}(\nabla\!\cdot\mathbf u^*)_c,\quad u_f = u^*_f - \tfrac{\Delta t}{h}\big(p_{n(f)} - p_{o(f)}\big) \qquad \mathrm{OSI} = \tfrac12\Big(1 - \tfrac{|\overline{\boldsymbol\tau}|}{\overline{|\boldsymbol\tau|}}\Big),\quad T_v = \min_u\big(T_u + \ell_{uv}/R_{uv}^2\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $o(f),\ n(f)$ | `mesh['owner'], mesh['neigh']` | (F,) | owner and neighbour cell of each voxel face, as in OpenFOAM's owner/neighbour lists |
| $u_f$ | `u` | (F,) | face-normal velocity (OpenFOAM's flux $\phi_f = u_f h^2$) |
| $(\nabla\!\cdot\mathbf u)_c$ | `divergence(u, owner, neigh, N, h)` | (N,) | two index_add calls over the faces: fvc::div(phi) |
| $A_{cd}$ | `pressure_matrix(mesh)['A']` | (M, M) sparse | $-\nabla^2$ assembled from the faces (fvm::laplacian); walls and inlet Neumann, outlets $p = 0$ |
| $A_{cd}\,p_d$ | `pcg(A, b, x, M, tol)` | (M,) | conjugate gradients, Jacobi plus a coarse-aggregate correction |
| $\nu\nabla^2 u_f$ | `face_laplacian(u, fnb, ghost, h)` | (F,) | six same-direction neighbours; ghost values put $u = 0$ exactly on the wall |
| $(\mathbf u\cdot\nabla)\mathbf u$ | `advect(u, mesh, dt)` | (F,) | semi-Lagrangian: trace each face back and interpolate (grid_sample) |
| $u_{\mathrm{in}}(r,t)$ | `womersley_shapes(r, H, omega, R, nu)` | (H+1, M) | Womersley profile per harmonic of $Q(t)$, Bessel $J_0, J_1$ of complex argument |
| $\boldsymbol\tau_w$ | `wall_shear(u1, u2, m, h, mu)` | (V, 3) | parabola through $u = 0$ at the wall and samples at $h$, $2h$ along the inward normal |
| $\overline{\vert \boldsymbol\tau\vert },\ \mathrm{OSI}$ | `tawss(tau), osi(tau)` | (V,) | averages over the recorded cycle, tau of shape (T, V, 3) |
| $R(\mathbf x)$ | `grid['phi']` | (nx, ny, nz) | distance to the wall = maximum-inscribed-sphere radius |
| $\dot{\mathbf x} = \mathbf u/\vert \mathbf u\vert $ | `streamlines(Uc, mesh, seeds)` | (S, L, 3) | batched RK4 streamlines through the trilinearly interpolated velocity, seeded at the inlet |
| $T_v$ | `relax(T, src, dst, w)` | (N,) | min-plus (Bellman–Ford) sweep of the eikonal equation on the 26-neighbour voxel graph |

Workflow. A typical aneurysm study segments the lumen, cleans it with VMTK (Antiga et al. 2008), solves the flow with OpenFOAM (Weller et al. 1998) and post-processes the wall shear stress. Here each step is ours, written with tensors; no code is copied from either GPL project. VMTK steps and their replacements: vmtksurfacesmoothing, vmtksurfaceremeshing and vmtkflowextensions (-adaptivelength 1 -extensionratio 5) were run once offline to make the STL files; vmtkcenterlines -seedselector profileidlist (Voronoi diagram, fast marching on $|\nabla T| = 1/R$, steepest-descent backtracking) becomes solve.centrelines: the inscribed radius $R$ is the wall distance of each voxel, the eikonal equation is solved by repeated min-plus sweeps over a 26-neighbour graph (a tensor stand-in for fast marching, with speed $R^2$ because the graph is the whole lumen rather than the already-medial Voronoi diagram), and all outlets backtrack along the predecessors at once. OpenFOAM's icoFoam/pimpleFoam and our code: lduAddressing owner/neighbour → mesh['owner'], mesh['neigh']; phi → u; fvc::div(phi) → divergence; fvm::laplacian(p) and its PCG solve → pressure_matrix and pcg; the PISO pressure correction → project (one Chorin projection per step); fvm::laplacian(nu, U) → face_laplacian; div(phi,U) → semi-Lagrangian advect; the wallShearStress function object → wss_at. Mesh: cubic voxels of the STL, with an immersed-boundary ghost value at the wall. Geometry: by default an aneurysm from the Lausanne TOF-MRA Aneurysm Cohort (Di Noto et al. 2022; OpenNeuro ds003949, CC0), subject 074: a 28 mm crop around the labelled aneurysm, thresholded, meshed with marching cubes, smoothed and flow-extended (data/SOURCES.md); one inlet and three outlets. The choice "synthetic" is our own CC0 bifurcation aneurysm. Other datasets checked: AneuriskWeb (CC BY-NC) and AneuX (CC BY-NC 4.0) are non-commercial; IntrA and AnXplore state no licence; the Vascular Model Repository has its own research licence; the Kiel phantom STLs (CC BY 4.0) are wall shells with very thin branches. Inflow: an analytic ICA-like waveform (mean 4.58 ml/s, peak/mean 1.66, Ford et al. 2005, period 1 s) scaled by Murray's law to the inlet radius, Womersley profile; outlets at zero pressure. Blood: Newtonian, $\nu = 3.5\times10^{-6}$ m²/s, $\rho = 1060$ kg/m³; rigid walls. Residuals (default run): pipe vs OpenFOAM v1912 pimpleFoam, axis velocity L2 1.2 %, phase −10 mrad, WSS L2 2.5 %; pipe vs Womersley's exact solution, L2 0.4 %; centrelines vs VMTK 1.5.0, mean distance 0.20 mm (0.5 voxel), maximum 0.5 mm, radius difference 0.12 mm on average; aneurysm vs pimpleFoam on our own voxel mesh exported as a polyMesh (same inflow and outlets), |u| on the slice L2 35 %, peak-systole WSS L2 38 %, TAWSS L2 32 %. That last gap is large: a run with a four times smaller time step gave the same, so it comes from the different wall treatment (our ghost values on a smooth surface against OpenFOAM's staircase faces), the outlet position and the advection schemes on a coarse mesh where neither code is converged. Read the aneurysm results as qualitative. The zero-pressure outlets also send too much flow into the smallest branch and give an unrealistic pressure drop. Output: besides the GIF, results/aneurysm.pvd is a ParaView time series of the wall (WSS, TAWSS, OSI), the streamlines and the tracers. Caveats: a coarse, educational simulation of a threshold segmentation, not for clinical use. Low WSS and high OSI in the sac are associated in the literature with aneurysm growth and rupture, but the evidence is conflicting and debated (Meng et al. 2014). The OpenFOAM and VMTK references were produced once, offline, and are never run by the problem or its tests.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/aneurysm_cfd/main.py
"""Main loop: STL → voxel mesh → centrelines → pulsatile Navier–Stokes → WSS, TAWSS, OSI → animation and 3D files,
with the residuals against the bundled OpenFOAM and VMTK reference results."""

import math
import time
from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation, plot, solve, tensors

TITLES = {
    "lausanne": "aneurysm lumen segmented from the CC0 Lausanne TOF-MRA cohort (sub-074)",
    "synthetic": "synthetic bifurcation aneurysm",
}


def pipe_check(nu: float, rho: float) -> dict:
    """Our solver on OpenFOAM's reference case: Womersley pipe, R = 2 mm, −∂_z p/ρ = 1.05 + 2.1 sin 2πt (m/s²)."""
    R = 2e-3
    grid = tensors.pipe_grid(R, R / 8)
    mesh = solve.build_mesh(grid, periodic=(False, False, True))
    out = solve.pipe_run(grid, mesh, 1.05, 2.1, 2 * math.pi, 2.0, 1e-3, nu, rho, n_rec=200)
    res = solve.openfoam_residuals(out, tensors.openfoam_reference(), 1.0, 2.0, rho)  # second cycle
    G = [complex(1.05), complex(0.0, -2.1)]
    exact = equation.womersley_pressure(torch.zeros(1, dtype=torch.float64), res["t"], G, 2 * math.pi, R, nu)[:, 0]
    sim = res["u_axis"][0]
    res["exact"] = exact
    res["an_L2"] = ((sim - exact).norm() / exact.norm()).item()
    ca, cb = solve.harmonic(sim, res["t"], 2 * math.pi), solve.harmonic(exact, res["t"], 2 * math.pi)
    res["an_amp"] = abs(ca) / abs(cb) - 1
    res["an_phase"] = math.remainder(math.atan2(ca.imag, ca.real) - math.atan2(cb.imag, cb.real), 2 * math.pi)
    return res


def aneurysm_residuals(rec: dict, ref: dict) -> dict:
    """Ours minus OpenFOAM on the same voxel mesh: |u| on the slice cells at the 25 sample times, |WSS| at the wall
    vertices at peak systole and TAWSS over the same samples. Relative L2 and L∞ errors."""
    du = rec["probe_u"].T - ref["u"]  # (S, T)
    kp = int((ref["t"] - ref["t_peak"]).abs().argmin())
    w_ours, ta_ours = rec["probe_wss"][kp], rec["probe_wss"].mean(0)

    def rel(d, r):
        return (d.norm() / r.norm()).item(), (d.abs().max() / r.abs().max()).item()

    out = {"du": du, "ref": ref, "dtawss": ta_ours - ref["tawss"], "dwss_peak": w_ours - ref["wss_peak"]}
    out["u_L2"], out["u_Linf"] = rel(du, ref["u"])
    out["wss_L2"], out["wss_Linf"] = rel(out["dwss_peak"], ref["wss_peak"])
    out["tawss_L2"], out["tawss_Linf"] = rel(out["dtawss"], ref["tawss"])
    return out


def run(params: dict, out_dir: Path) -> Path:
    torch.manual_seed(0)
    geometry = params["geometry"]
    h = params["voxel"] * tensors.MM
    nu, rho = tensors.NU, tensors.RHO
    t0 = time.time()
    tris, ids = tensors.surface(geometry)
    grid = tensors.lumen_grid(tris, ids, h)
    mesh = solve.build_mesh(grid)
    vol = tensors.mesh_volume(tris)
    print(f"{geometry} surface: {len(tris)} triangles, {vol * 1e9:.0f} mm³;  voxels h = {h * 1e3:.2f} mm: "
          f"{mesh['n']} fluid cells ({mesh['n'] * h**3 * 1e9:.0f} mm³), {len(mesh['owner'])} faces")  # fmt: skip
    caps = grid["caps"]
    print("caps (after VMTK flow extensions):  inlet r = %.2f mm,  outlets r = %s mm" % (
        caps[0]["radius"] * 1e3, ", ".join(f"{c['radius'] * 1e3:.2f}" for c in caps[1:])))  # fmt: skip

    # centrelines: tensor version of vmtkcenterlines, with the residual against the bundled VMTK result
    seeds = torch.tensor([solve.outlet_seed(grid, c) for c in caps[1:]])
    lines, bif, info = solve.centrelines(grid, solve.cap_nodes(grid, caps[0]), seeds)
    vres = solve.vmtk_residuals(lines, *tensors.vmtk_reference(geometry))
    print(f"eikonal: {info['iters']} min-plus sweeps;  bifurcation at abscissa {bif['s'] * 1e3:.1f} mm")
    for r in vres:
        d, dR, line = r["d"], r["dR"], lines[r["k"]]
        print(f"centreline to outlet {r['k'] + 1}: {line['s'][-1] * 1e3:.1f} mm (VMTK {r['s_ref'][-1] * 1e3:.1f}): "
              f"distance mean {d.mean() * 1e3:.3f} / max {d.max() * 1e3:.3f} mm "
              f"({d.mean() / h:.2f} / {d.max() / h:.2f} voxels),  |ΔR| mean {dR.mean() * 1e3:.3f} / "
              f"max {dR.max() * 1e3:.3f} mm ({dR.mean() / h:.2f} / {dR.max() / h:.2f} voxels)")  # fmt: skip

    # the aneurysm sac: lumen voxels farther from every centreline than its local radius (+ ½ mm)
    allx, allR = torch.cat([ln["x"] for ln in lines]), torch.cat([ln["R"] for ln in lines])
    dmin, j = torch.cdist(mesh["pos"], allx).min(1)
    sac = dmin > allR[j] + 0.5e-3
    print(f"aneurysm sac (lumen off every centreline): {sac.sum() * h**3 * 1e9:.0f} mm³")

    # residuals of the flow solver against OpenFOAM (pimpleFoam) and Womersley's exact solution on the pipe case
    pres = pipe_check(nu, rho)
    print(f"pipe vs OpenFOAM v1912 pimpleFoam (cycle 2): u_axis L2 {pres['u_axis_L2'] * 100:.2f} %, "
          f"L∞ {pres['u_axis_Linf'] * 100:.2f} %, amplitude {pres['amp_err'] * 100:+.2f} %, "
          f"phase {pres['phase_err'] * 1e3:+.1f} mrad;  WSS L2 {pres['wss_L2'] * 100:.2f} %, "
          f"L∞ {pres['wss_Linf'] * 100:.2f} %;  flow rate L2 {pres['Q_L2'] * 100:.2f} %")  # fmt: skip
    print(
        f"pipe vs Womersley (exact): u_axis L2 {pres['an_L2'] * 100:.2f} %, amplitude {pres['an_amp'] * 100:+.2f} %, "
        f"phase {pres['an_phase'] * 1e3:+.1f} mrad;  Δp imposed in both (the driver)"
    )

    slice_y = int(round(((mesh["pos"][sac].mean(0)[1] - grid["lo"][1]) / h).item()))  # slice through the sac
    t_wave, Q = tensors.inflow_waveform()
    R0 = caps[0]["radius"]
    Q = Q * (R0 / tensors.R_ICA) ** 3  # Murray's law: scale the ICA flow to the inlet radius
    dt = params["dt_ms"] * 1e-3
    lam = dt * nu / h**2 * (12 + 4 * (1 - mesh["ghost"].min().item()))
    print(f"mean inflow {Q.mean() * 1e6:.2f} ml/s (4.58 ml/s × (R_in/2 mm)³);  Δt = {dt * 1e3:.2f} ms "
          f"(explicit viscous number {lam:.2f} < 2;  semi-Lagrangian advection has no CFL limit)")  # fmt: skip
    ref = tensors.openfoam_aneurysm_reference() if geometry == "lausanne" else None
    probe = None if ref is None else {"times": ref["t"], "points": ref["points"]}
    rec = solve.simulate(mesh, (t_wave, Q), params["cycles"], dt, nu, rho, grid["V"], grid["N"],
                         n_frames=params["frames"], slice_at=slice_y, probe=probe)  # fmt: skip
    q_in, q_out = torch.tensor(rec["Q_in"]), rec["Q_out"]
    err = ((q_out.sum(1) - q_in).abs().max() / q_in.abs().max()).item()
    split = " / ".join(f"{(q_out[:, k].mean() / q_in.mean()) * 100:.1f} %" for k in range(q_out.shape[1]))
    print(
        f"mass: inflow {q_in.mean() * 1e6:.3f} ml/s (prescribed {Q.mean() * 1e6:.3f}), outflow split {split}, "
        f"max |in − Σout|/in = {err:.1e};  PCG iterations/step {sum(rec['iters']) / len(rec['iters']):.1f}"
    )
    dp = torch.tensor(rec["dp"])
    print(f"pressure drop inlet → outlets: mean {dp.mean():.0f} Pa ({dp.mean() / 133.3:.1f} mmHg), "
          f"range {dp.min():.0f} … {dp.max():.0f} Pa over the cycle")  # fmt: skip

    ares = None
    if ref is not None and len(rec["probe_t"]) == len(ref["t"]):
        ares = aneurysm_residuals(rec, ref)
        print(f"aneurysm vs OpenFOAM (same voxel mesh): |u| on the slice L2 {ares['u_L2'] * 100:.1f} %, "
              f"L∞ {ares['u_Linf'] * 100:.1f} %;  |WSS| at peak systole L2 {ares['wss_L2'] * 100:.1f} %, "
              f"L∞ {ares['wss_Linf'] * 100:.1f} %;  TAWSS L2 {ares['tawss_L2'] * 100:.1f} %, "
              f"L∞ {ares['tawss_Linf'] * 100:.1f} %")  # fmt: skip
    else:
        print("aneurysm vs OpenFOAM: no reference for the synthetic geometry (skipped)")

    tau = rec["wss"]  # (T, V, 3)
    TA, OS = equation.tawss(tau), equation.osi(tau)
    vsac = torch.cdist(grid["V"].float(), mesh["pos"][sac].float()).min(1).values < h  # wall vertices on the sac
    print(f"TAWSS: parent/branches median {TA[~vsac].median():.2f} Pa, aneurysm sac median {TA[vsac].median():.2f} Pa")
    print(f"OSI: parent/branches median {OS[~vsac].median():.3f}, sac median {OS[vsac].median():.3f}, "
          f"sac max {OS[vsac].max():.3f}")  # fmt: skip
    print(f"parent artery: Poiseuille 4μQ̄/(πR³) = {equation.poiseuille_wss(Q.mean().item(), R0, rho * nu):.2f} Pa, "
          f"Reynolds 2Q̄/(πRν) = {2 * Q.mean().item() / (math.pi * R0 * nu):.0f}, "
          f"Womersley α = R√(ω/ν) = {R0 * math.sqrt(2 * math.pi / rec['T'] / nu):.2f}")  # fmt: skip

    # interactive 3D results for the web page: an animated VTK collection (surface, streamlines, particles)
    pvd = plot.write_vtk_series(out_dir / "results", tris, ids, grid, rec, TA, OS)
    size = sum(f.stat().st_size for f in pvd.parent.iterdir()) / 1e6
    print(f"ParaView time series: {pvd.relative_to(out_dir)} with {len(rec['t'])} steps × 3 layers, {size:.1f} MB")
    print(f"compute time {time.time() - t0:.1f} s (animation next)")
    return plot.animate(tris, ids, grid, mesh, lines, vres, rec, TA, OS, sac, pres, ares, (t_wave, Q),
                        out_dir / "aneurysm_cfd.gif", title=TITLES[geometry])  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/aneurysm_cfd/equation.py
r"""Physics → tensor translation: incompressible Navier–Stokes on a voxel finite-volume mesh, plus the vessel laws.

The laws (index form, repeated j summed):

    ∂_t u_i + u_j ∂_j u_i = -∂_i p/ρ + ν ∂_j ∂_j u_i + g_i,        ∂_j u_j = 0

On the mesh every quantity lives where OpenFOAM keeps it: pressure p_c in cells c, the face-normal velocity
u_f on faces f (OpenFOAM's flux φ_f = u_f A_f). Each internal face has an owner cell o(f) and a neighbour cell
n(f) (OpenFOAM's lduAddressing). Then

    divergence      (∇·u)_c = (Σ_{f: o(f)=c} u_f − Σ_{f: n(f)=c} u_f) / h → index_add over faces (fvc::div(phi))
    Laplacian       (∇²p)_c = Σ_{f ∋ c} ±(p_n(f) − p_o(f)) / h² → index_add over faces (fvm::laplacian)
    gradient        (∇p)_f = (p_n(f) − p_o(f)) / h → gather (fvc::snGrad)

Chorin's projection makes one time step: u* = advect(u) + Δt(ν∇²u + g), then solve ∇²p = ∇·u*/Δt and set
u = u* − Δt ∇p, which leaves ∇·u = 0 (to the tolerance of the conjugate-gradient solve).

Vessel laws used as checks: Hagen–Poiseuille, Womersley's oscillatory pipe flow, wall shear stress τ = μ ∂u_t/∂n,
its time average (TAWSS) and the oscillatory shear index (OSI). For the centreline: the eikonal equation
|∇T| = 1/R^p solved by min-plus relaxation T_v = min_u (T_u + ℓ_uv / R_uv^p) on the voxel graph.
"""

import math

import torch


# ---------------------------------------------------------------- finite-volume operators on the face/cell lists
def divergence(u: torch.Tensor, owner: torch.Tensor, neigh: torch.Tensor, n_cells: int, h: float) -> torch.Tensor:
    r"""(∇·u)_c = (Σ_{o(f)=c} u_f − Σ_{n(f)=c} u_f)/h.   u: (F,) face velocities → (N,) per cell (1/s)."""
    out = torch.zeros(n_cells, dtype=u.dtype, device=u.device)
    return out.index_add(0, owner, u).index_add(0, neigh, -u) / h


def laplacian(p: torch.Tensor, owner: torch.Tensor, neigh: torch.Tensor, h: float) -> torch.Tensor:
    r"""(∇²p)_c = Σ_f ±(p_{n(f)} − p_{o(f)})/h² over the given faces.   p: (N,) → (N,)."""
    flux = (p[neigh] - p[owner]) / h**2  # (F,) one value per face, added to the owner and taken from the neighbour
    return torch.zeros_like(p).index_add(0, owner, flux).index_add(0, neigh, -flux)


def face_gradient(p: torch.Tensor, owner: torch.Tensor, neigh: torch.Tensor, h: float) -> torch.Tensor:
    r"""(∇p)_f = (p_{n(f)} − p_{o(f)})/h, the face-normal gradient.   (N,) → (F,)."""
    return (p[neigh] - p[owner]) / h


def face_laplacian(u: torch.Tensor, fnb: torch.Tensor, ghost: torch.Tensor, h: float) -> torch.Tensor:
    r"""ν∇²u for face velocities: (Σ_s u_{s} − 6 u_f)/h² over the 6 same-direction neighbours s.

    fnb: (F, 6) neighbour face index or −1 (a wall). A missing neighbour takes the ghost value g_fs u_f, chosen
    so that the velocity is 0 exactly on the wall (the immersed-boundary ghost-cell method).
    """
    nbv = torch.where(fnb >= 0, u[fnb.clamp(min=0)], ghost * u[:, None])  # (F, 6)
    return (nbv.sum(1) - 6 * u) / h**2


# ---------------------------------------------------------------- vessel laws (analytic checks)
def bessel_j(nu: int, z: torch.Tensor, terms: int = 60) -> torch.Tensor:
    r"""J_ν(z) = Σ_k (−1)^k (z/2)^{2k+ν} / (k! (k+ν)!) for complex z (power series, fine for |z| ≲ 20)."""
    k = torch.arange(terms, dtype=torch.float64, device=z.device)
    log_fact = torch.lgamma(k + 1) + torch.lgamma(k + nu + 1)  # (K,)
    zz = (z[..., None] / 2) ** (2 * k + nu)  # (..., K)
    return (zz * ((-1.0) ** k * torch.exp(-log_fact))).sum(-1)


def poiseuille_wss(Q: float, R: float, mu: float) -> float:
    r"""Hagen–Poiseuille wall shear stress τ_w = 4μQ/(πR³)."""
    return 4 * mu * Q / (math.pi * R**3)


def womersley_pressure(r: torch.Tensor, t: torch.Tensor, G: list, omega: float, R: float, nu: float) -> torch.Tensor:
    r"""Axial velocity driven by −∂_z p/ρ = Re Σ_n G_n e^{i n ω t} (G_0 steady):

    u = G_0 (R² − r²)/(4ν) + Re Σ_{n≥1} (G_n/(i n ω)) (1 − J_0(Λ_n r/R)/J_0(Λ_n)) e^{i n ω t},  Λ_n = i^{3/2} R √(nω/ν)
    r: (M,), t: (T,) → (T, M).
    """
    u = (G[0].real * (R**2 - r**2) / (4 * nu))[None, :].repeat(len(t), 1)
    for n, Gn in enumerate(G[1:], start=1):
        lam = complex(math.cos(3 * math.pi / 4), math.sin(3 * math.pi / 4)) * R * math.sqrt(n * omega / nu)
        lam_t = torch.tensor(lam, dtype=torch.complex128)
        shape = 1 - bessel_j(0, lam_t * r.to(torch.complex128) / R) / bessel_j(0, lam_t)  # (M,)
        phase = torch.exp(1j * n * omega * t.to(torch.complex128))  # (T,)
        u = u + (Gn / (1j * n * omega) * phase[:, None] * shape[None, :]).real
    return u


def womersley_shapes(r: torch.Tensor, n_harm: int, omega: float, R: float, nu: float) -> torch.Tensor:
    r"""Womersley's profile per unit flow for each harmonic n = 0 … H (each carries exactly 1 m³/s):

    S_0 = (2/πR²)(1 − r²/R²),   S_n = (1/πR²) (1 − J_0(Λ_n r/R)/J_0(Λ_n)) / (1 − 2J_1(Λ_n)/(Λ_n J_0(Λ_n)))
    r: (M,) → (H+1, M) complex.
    """
    area = math.pi * R**2
    rr = (r / R).clamp(max=1.0).to(torch.complex128)
    out = [2 / area * (1 - rr**2)]
    for n in range(1, n_harm + 1):
        lam = torch.tensor(
            complex(math.cos(3 * math.pi / 4), math.sin(3 * math.pi / 4)) * R * math.sqrt(n * omega / nu)
        )
        j0 = bessel_j(0, lam)
        out.append((1 - bessel_j(0, lam * rr) / j0) / (1 - 2 * bessel_j(1, lam) / (lam * j0)) / area)
    return torch.stack(out)


def womersley_flow(r: torch.Tensor, t: torch.Tensor, Qn: torch.Tensor, omega: float, R: float, nu: float):
    r"""Axial velocity for a given flow rate Q(t) = Re Σ_n Q_n e^{inωt}: u(r, t) = Re Σ_n Q_n S_n(r) e^{inωt}.
    r: (M,), t: (T,), Qn: (H+1,) complex → (T, M)."""
    S = womersley_shapes(r, len(Qn) - 1, omega, R, nu)  # (H+1, M)
    n = torch.arange(len(Qn), dtype=torch.float64)
    phase = torch.exp(1j * omega * t.to(torch.float64)[:, None] * n[None, :])  # (T, H+1)
    return torch.einsum("th,h,hm->tm", phase, Qn.to(torch.complex128), S).real


# ---------------------------------------------------------------- wall shear stress and its indices
def wall_shear(u1: torch.Tensor, u2: torch.Tensor, m: torch.Tensor, delta: float, mu: float) -> torch.Tensor:
    r"""τ_i = μ (δ_ij − m_i m_j) ∂_n u_j at the wall, with ∂_n u from a parabola through u = 0 at the wall and the
    samples u1 = u(δ), u2 = u(2δ) along the inward normal m: ∂_n u = (4 u1 − u2)/(2δ).   (V, 3) each → (V, 3)."""
    a = (4 * u1 - u2) / (2 * delta)
    return mu * (a - torch.einsum("vj,vj->v", a, m)[:, None] * m)


def tawss(tau: torch.Tensor) -> torch.Tensor:
    r"""TAWSS = (1/T)∫|τ| dt.   tau: (T, V, 3) → (V,)."""
    return tau.norm(dim=-1).mean(0)


def osi(tau: torch.Tensor) -> torch.Tensor:
    r"""OSI = ½ (1 − |∫τ dt| / ∫|τ| dt): 0 for one-way shear, ½ for purely oscillating shear.   (T, V, 3) → (V,)."""
    return 0.5 * (1 - tau.mean(0).norm(dim=-1) / tau.norm(dim=-1).mean(0).clamp(min=1e-12))


# ---------------------------------------------------------------- centreline: eikonal by min-plus relaxation
def edge_cost(length: torch.Tensor, R_mid: torch.Tensor, power: float) -> torch.Tensor:
    r"""Travel time along a graph edge, ℓ / R^p: the front moves fastest where the inscribed sphere is largest."""
    return length / R_mid.clamp(min=1e-9) ** power


def relax(T: torch.Tensor, src: torch.Tensor, dst: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""One min-plus (Bellman–Ford) sweep of the discrete eikonal equation: T_v ← min(T_v, min_u T_u + w_uv)."""
    return T.scatter_reduce(0, dst, T[src] + w, reduce="amin", include_self=True)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/aneurysm_cfd/tensors.py
"""Build the concrete tensors: read the STL and CSV data, voxelise the lumen, signed distance to the wall.

data/aneurysm.stl          watertight surface in mm; the 2-byte STL attribute of each triangle is 0 wall, 1 inlet cap,
                           2, 3 outlet caps (synthetic geometry, made offline with VMTK: see meta.yaml notes)
data/inflow_waveform.csv   Q(t) in ml/s over one cardiac cycle (analytic ICA-like waveform, Ford et al. 2005 mean)
data/vmtk_reference_centerlines.csv   VMTK centrelines of the same surface (test reference only)
data/openfoam_reference_womersley.csv OpenFOAM pimpleFoam pipe run (test reference only)
"""

import math
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data"
RHO, NU = 1060.0, 3.5e-6  # blood density (kg/m³) and kinematic viscosity (m²/s), Newtonian
MM = 1e-3


# ---------------------------------------------------------------- file readers (no VTK, no external tools)
def read_stl(path: Path) -> tuple[torch.Tensor, torch.Tensor]:
    """Binary STL → triangles (F, 3, 3) in metres and the 2-byte attribute per triangle (F,)."""
    raw = Path(path).read_bytes()
    n = int(np.frombuffer(raw[80:84], np.uint32)[0])
    rec = np.frombuffer(raw[84 : 84 + 50 * n], np.dtype([("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")]))
    tris = torch.tensor(rec["v"].astype(np.float64)) * MM
    return tris, torch.tensor(rec["a"].astype(np.int64))


def read_csv(path: Path) -> np.ndarray:
    """A numeric CSV with '#' comment lines and one header line."""
    lines = [s for s in Path(path).read_text().splitlines() if s and not s.startswith("#")]
    return np.array([[float(x) for x in s.split(",")] for s in lines[1:]])


def inflow_waveform() -> tuple[torch.Tensor, torch.Tensor]:
    """t (K,) in s and Q (K,) in m³/s over one period."""
    a = read_csv(DATA / "inflow_waveform.csv")
    return torch.tensor(a[:, 0]), torch.tensor(a[:, 1]) * 1e-6


GEOMETRIES = {  # surface STL and its VMTK reference centrelines
    "lausanne": ("aneurysm_lausanne.stl", "vmtk_reference_centerlines_lausanne.csv"),
    "synthetic": ("aneurysm.stl", "vmtk_reference_centerlines.csv"),
}
R_ICA = 2.0e-3  # internal carotid radius that the 4.58 ml/s mean flow belongs to (m)


def surface(name: str = "lausanne") -> tuple[torch.Tensor, torch.Tensor]:
    """Triangles (F, 3, 3) in m and cap ids (0 wall, 1 inlet, 2… outlets) of a bundled geometry."""
    return read_stl(DATA / GEOMETRIES[name][0])


def vmtk_reference(name: str = "lausanne") -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """VMTK centreline points (M, 3) in m, maximum-inscribed-sphere radius (M,) in m, outlet index (M,)."""
    a = torch.tensor(read_csv(DATA / GEOMETRIES[name][1]))
    return a[:, :3] * MM, a[:, 3] * MM, a[:, 4].long()


def openfoam_reference() -> dict:
    a = torch.tensor(read_csv(DATA / "openfoam_reference_womersley.csv"))
    return {"t": a[:, 0], "u_axis": a[:, 1], "u_half": a[:, 2], "wss": a[:, 3], "Q": a[:, 4]}


# ---------------------------------------------------------------- geometry helpers
def mesh_volume(tris: torch.Tensor) -> float:
    """Signed volume ⅙ Σ a·(b×c) of a closed, outward-oriented triangle mesh (divergence theorem)."""
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    return (torch.einsum("fi,fi->f", a, torch.linalg.cross(b, c)).sum() / 6).item()


def cylinder_mesh(R: float, L: float, n_theta: int = 64, n_z: int = 20) -> tuple[torch.Tensor, torch.Tensor]:
    """Closed triangulated cylinder along z from 0 to L (outward normals); ids 0 wall, 1 cap at z=0, 2 cap at z=L."""
    th = torch.linspace(0, 2 * math.pi, n_theta + 1, dtype=torch.float64)[:-1]
    z = torch.linspace(0, L, n_z + 1, dtype=torch.float64)
    ring = torch.stack([R * torch.cos(th), R * torch.sin(th)], -1)  # (Θ, 2)
    P = torch.cat([ring[None].expand(n_z + 1, -1, -1), z[:, None, None].expand(-1, n_theta, 1)], -1)  # (Z+1, Θ, 3)
    i, j = torch.meshgrid(torch.arange(n_z), torch.arange(n_theta), indexing="ij")
    j1 = (j + 1) % n_theta
    a, b, c, d = P[i, j], P[i, j1], P[i + 1, j1], P[i + 1, j]
    wall = torch.cat([torch.stack([a, b, c], -2), torch.stack([a, c, d], -2)]).reshape(-1, 3, 3)
    centre = lambda zz: torch.tensor([0.0, 0.0, zz], dtype=torch.float64)  # noqa: E731
    bot = torch.stack([centre(0.0).expand(n_theta, 3), P[0, (torch.arange(n_theta) + 1) % n_theta], P[0]], 1)
    top = torch.stack([centre(L).expand(n_theta, 3), P[-1], P[-1, (torch.arange(n_theta) + 1) % n_theta]], 1)
    tris = torch.cat([wall, bot, top])
    ids = torch.cat([torch.zeros(len(wall)), torch.ones(n_theta), 2 * torch.ones(n_theta)]).long()
    return tris, ids


def voxelize(tris: torch.Tensor, lo: torch.Tensor, h: float, shape: tuple) -> torch.Tensor:
    """Inside test by ray parity: a voxel centre is inside if a ray along +z crosses the surface an odd number of times.

    For each triangle and each (x, y) ray column under it, the crossing height z comes from barycentric coordinates;
    crossings are histogrammed into the z cells and a cumulative sum mod 2 fills the lumen. → (nx, ny, nz) bool.
    """
    nx, ny, nz = shape
    P = (tris - lo) / h  # (F, 3, 3) in voxel units; voxel centre i sits at i
    mn, mx = P[..., :2].min(1).values.ceil().long(), P[..., :2].max(1).values.floor().long()  # (F, 2) ray columns
    k = int((mx - mn).max().item()) + 1
    off = torch.stack(torch.meshgrid(torch.arange(k), torch.arange(k), indexing="ij"), -1).reshape(-1, 2)  # (K, 2)
    cand = mn[:, None, :] + off[None]  # (F, K, 2) candidate columns
    f, c = (cand <= mx[:, None, :]).all(-1).nonzero(as_tuple=True)
    q = cand[f, c].double() + torch.tensor([1e-7, 1.4142e-7], dtype=torch.float64)  # tiny shift: never hit an edge
    a, b, cc = P[f, 0], P[f, 1], P[f, 2]  # (M, 3) each
    e1, e2, e3 = b[:, :2] - a[:, :2], cc[:, :2] - a[:, :2], q - a[:, :2]
    det = e1[:, 0] * e2[:, 1] - e1[:, 1] * e2[:, 0]
    s = (e3[:, 0] * e2[:, 1] - e3[:, 1] * e2[:, 0]) / det
    t = (e1[:, 0] * e3[:, 1] - e1[:, 1] * e3[:, 0]) / det
    hit = (s >= 0) & (t >= 0) & (s + t <= 1) & (det.abs() > 1e-14)
    z = (a[:, 2] + s * (b[:, 2] - a[:, 2]) + t * (cc[:, 2] - a[:, 2]))[hit]
    col = q[hit].round().long()
    iz = (torch.floor(z).long() + 1).clamp(0, nz)  # the crossing lies between centres floor(z) and floor(z) + 1
    keep = (col[:, 0] >= 0) & (col[:, 0] < nx) & (col[:, 1] >= 0) & (col[:, 1] < ny)
    count = torch.zeros(nx, ny, nz + 1, dtype=torch.long)
    count.index_put_((col[keep, 0], col[keep, 1], iz[keep]), torch.ones_like(iz[keep]), accumulate=True)
    return (count.cumsum(2)[..., :nz] % 2) == 1


def vertices_and_normals(tris: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Unique vertices (V, 3) and area-weighted outward unit normals (V, 3) of a triangle soup."""
    V, inv = torch.unique((tris.reshape(-1, 3) / 1e-9).round(), dim=0, return_inverse=True)
    V = V * 1e-9
    fn = torch.linalg.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])  # (F, 3), |fn| = 2 area
    N = torch.zeros_like(V).index_add(0, inv, fn.repeat_interleave(3, 0))
    return V, N / N.norm(dim=1, keepdim=True).clamp(min=1e-30)


def wall_distance(x: torch.Tensor, V: torch.Tensor, N: torch.Tensor, chunk: int = 4096) -> torch.Tensor:
    """Distance from points x (M, 3) to the wall: |n_v·(x − v)| for the nearest wall vertex v (tangent plane)."""
    out = []
    for i in range(0, len(x), chunk):
        xi = x[i : i + chunk]
        k = torch.cdist(xi.float(), V.float()).argmin(1)  # (m,) nearest vertex
        out.append(torch.einsum("mi,mi->m", xi - V[k], N[k]).abs())
    return torch.cat(out) if out else x.new_zeros(0)


def caps(tris: torch.Tensor, ids: torch.Tensor) -> list[dict]:
    """For each cap id ≥ 1: centre, outward unit normal and equivalent radius √(A/π)."""
    out = []
    for k in range(1, int(ids.max()) + 1):
        t = tris[ids == k]
        fn = torch.linalg.cross(t[:, 1] - t[:, 0], t[:, 2] - t[:, 0]) / 2  # (F, 3) area vectors
        area = fn.norm(dim=1)
        centre = (t.mean(1) * area[:, None]).sum(0) / area.sum()
        n = fn.sum(0)
        out.append({"centre": centre, "normal": n / n.norm(), "radius": math.sqrt(area.sum().item() / math.pi)})
    return out


def lumen_grid(tris: torch.Tensor, ids: torch.Tensor, h: float, pad: int = 3) -> dict:
    """Voxelise the lumen and compute the signed wall distance φ (> 0 in the fluid) on the cell centres.

    φ is exact enough near the wall (tangent plane of the nearest wall vertex, cap triangles excluded) and is the
    maximum-inscribed-sphere radius R(x) deep inside, which the centreline step uses.
    """
    P = tris.reshape(-1, 3)
    lo = P.min(0).values - pad * h
    shape = tuple(int(s) for s in ((P.max(0).values + pad * h - lo) / h).ceil().long().tolist())
    inside = voxelize(tris, lo, h, shape)
    V, N = vertices_and_normals(tris[ids == 0])
    band = torch.nn.functional.max_pool3d(inside.float()[None, None], 5, 1, 2)[0, 0] > 0  # fluid + 2 solid layers
    idx = band.nonzero()
    phi = torch.full(shape, -3 * h, dtype=torch.float64)
    d = wall_distance(lo + h * idx.double(), V, N)
    phi[tuple(idx.T)] = torch.where(inside[tuple(idx.T)], d, -d)
    return {"lo": lo, "h": h, "shape": shape, "inside": inside, "phi": phi, "V": V, "N": N, "caps": caps(tris, ids)}


def pipe_grid(R: float, h: float, nz: int = 2) -> dict:
    """Straight periodic pipe along z with the exact signed distance φ = R − r (used for the Poiseuille/Womersley
    checks); the wall sample points are a ring of vertices at mid-height with outward normals."""
    n = int(math.ceil(R / h)) + 3
    ax = h * (torch.arange(2 * n + 1, dtype=torch.float64) - n)
    X, Y = torch.meshgrid(ax, ax, indexing="ij")
    phi = (R - torch.sqrt(X**2 + Y**2))[:, :, None].expand(-1, -1, nz).clone()
    th = torch.linspace(0, 2 * math.pi, 65, dtype=torch.float64)[:-1]
    N = torch.stack([torch.cos(th), torch.sin(th), torch.zeros_like(th)], -1)
    V = R * N + torch.tensor([0.0, 0.0, h * (nz - 1) / 2], dtype=torch.float64)
    lo = torch.tensor([-n * h, -n * h, 0.0], dtype=torch.float64)
    return {"lo": lo, "h": h, "shape": tuple(phi.shape), "inside": phi > 0, "phi": phi, "V": V, "N": N, "caps": []}


def tube_mask(polylines: list, radii: list, h: float, pad: float = 3.0) -> dict:
    """Voxel mask and exact inscribed radius R = −sdf of a union of tubes around polylines (centreline checks)."""
    pts = torch.cat([torch.tensor(p, dtype=torch.float64) for p in polylines])
    rmax = max(radii)
    lo = pts.min(0).values - rmax - pad * h
    shape = tuple(int(s) for s in ((pts.max(0).values + rmax + pad * h - lo) / h).ceil().long().tolist())
    axes = [lo[i] + h * torch.arange(shape[i], dtype=torch.float64) for i in range(3)]
    X = torch.stack(torch.meshgrid(*axes, indexing="ij"), -1).reshape(-1, 3)
    sdf = torch.full((len(X),), float("inf"), dtype=torch.float64)
    for poly, r in zip(polylines, radii):
        p = torch.tensor(poly, dtype=torch.float64)
        a, b = p[:-1], p[1:]  # (S, 3) segments
        ab = b - a
        t = (torch.einsum("msi,si->ms", X[:, None] - a, ab) / (ab * ab).sum(1)).clamp(0, 1)
        d = (X[:, None] - a - t[..., None] * ab).norm(dim=-1).min(1).values
        sdf = torch.minimum(sdf, d - r)
    phi = (-sdf).reshape(shape)
    return {"lo": lo, "h": h, "shape": shape, "inside": phi > 0, "phi": phi, "caps": []}


def openfoam_aneurysm_reference() -> dict | None:
    """OpenFOAM pimpleFoam on our own voxel mesh of the Lausanne geometry at h = 0.4 mm (see the CSV headers):
    |u| at slice points (cell centres) at 25 times in the last cycle, and |WSS| at the wall vertices (peak systole,
    TAWSS). Our run is sampled at the same points (trilinear) and vertices, so any voxel size can be compared."""
    f_sl, f_w = DATA / "openfoam_reference_aneurysm_slice.csv", DATA / "openfoam_reference_aneurysm_wss.csv"
    if not (f_sl.is_file() and f_w.is_file()):
        return None
    head = [s for s in f_sl.read_text().splitlines() if s and not s.startswith("#")][0].split(",")
    a, w = torch.tensor(read_csv(f_sl)), torch.tensor(read_csv(f_w))
    t = torch.tensor([float(c[3:]) for c in head[6:]], dtype=torch.float64)
    peak = float([s for s in f_w.read_text().splitlines() if "peak systole" in s][0].split("t = ")[1].split(" s")[0])
    return {"cells": a[:, :3].long(), "points": a[:, 3:6] * MM, "t": t, "u": a[:, 6:], "V": w[:, :3] * MM,
            "wss_peak": w[:, 3],
            "tawss": w[:, 4], "t_peak": peak}  # fmt: skip


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/aneurysm_cfd/solve.py
"""The solver: voxel finite-volume mesh (owner/neighbour lists), Chorin projection with a conjugate-gradient
pressure solve, Womersley inflow, wall shear stress, tracer particles, and the VMTK-style centreline."""

import math

import torch
import torch.nn.functional as Fn

from problems._common.progress import steps

from . import equation as eq

DIRS = [(0, -1), (0, 1), (1, -1), (1, 1), (2, -1), (2, 1)]  # neighbour slot s = 2·axis + (step > 0)


# ---------------------------------------------------------------- mesh: cells, faces, neighbour tables
def shift(a: torch.Tensor, axis: int, step: int, periodic: bool, fill) -> torch.Tensor:
    """out[i] = a[i + step] along axis; `fill` where that falls outside a non-periodic box."""
    out = torch.roll(a, -step, axis)
    if not periodic:
        sl = [slice(None)] * a.dim()
        sl[axis] = slice(-step, None) if step > 0 else slice(0, -step)
        out[tuple(sl)] = fill
    return out


def build_mesh(grid: dict, periodic=(False, False, False)) -> dict:
    """Fluid cells, faces between fluid neighbours (owner → neighbour along +x, +y, +z), face neighbour tables with
    ghost factors for the no-slip wall, and cell kinds from the caps: 0 interior, 1 inlet, 2 outlet."""
    inside, h, lo, phi = grid["inside"], grid["h"], grid["lo"], grid["phi"]
    shape = inside.shape
    n = int(inside.sum())
    idx = torch.full(shape, -1, dtype=torch.long)
    idx[inside] = torch.arange(n)
    cells = inside.nonzero()  # (N, 3) grid indices, same order as idx
    pos = lo + h * cells.double()  # (N, 3) cell centres (m)
    nb = torch.stack([shift(idx, a, s, periodic[a], -1)[inside] for a, s in DIRS], 1)  # (N, 6)

    owner, neigh, dirn = [], [], []
    face_of = torch.full((n, 3), -1, dtype=torch.long)
    for d in range(3):
        o = (nb[:, 2 * d + 1] >= 0).nonzero()[:, 0]
        start = sum(len(x) for x in owner)
        face_of[o, d] = torch.arange(start, start + len(o))
        owner.append(o), neigh.append(nb[o, 2 * d + 1]), dirn.append(torch.full_like(o, d))
    owner, neigh, dirn = torch.cat(owner), torch.cat(neigh), torch.cat(dirn)
    e = torch.eye(3, dtype=torch.float64)
    fpos = pos[owner] + 0.5 * h * e[dirn]  # (F, 3) face centres

    # same-direction neighbour faces; a missing one is a wall: 0 across it (normal) or a ghost value (tangential)
    phi_at = lambda g: phi[tuple(g.T)]  # noqa: E731  φ at integer grid indices (F, 3)
    gmax = torch.tensor(shape) - 1
    wrap = lambda g: torch.where(torch.tensor(periodic), g % torch.tensor(shape), g.clamp(min=0).minimum(gmax))  # noqa: E731
    phi_f = 0.5 * (phi_at(cells[owner]) + phi_at(cells[neigh]))  # (F,)
    fnb = torch.full((len(owner), 6), -1, dtype=torch.long)
    ghost = torch.zeros(len(owner), 6, dtype=torch.float64)
    for s, (a, sg) in enumerate(DIRS):
        o2 = nb[owner, s]
        fnb[:, s] = torch.where(o2 >= 0, face_of[o2.clamp(min=0), dirn], -1)
        off = (sg * e[a]).long()
        phi_g = 0.5 * (phi_at(wrap(cells[owner] + off)) + phi_at(wrap(cells[owner] + off + e[dirn].long())))
        theta = (phi_f / (phi_f - phi_g).clamp(min=1e-12)).clamp(0.25, 1.0)  # wall at θh from the face centre
        ghost[:, s] = torch.where(dirn == a, 0.0, 1 - 1 / theta)  # linear profile through u = 0 at the wall

    # the 4 faces of each other direction e around face f (both sides of owner and neighbour): (F, 3, 4), −1 = wall
    def faces_e(c, e_):
        minus = torch.where(nb[c, 2 * e_] >= 0, face_of[nb[c, 2 * e_].clamp(min=0), e_], -1)
        return torch.stack([face_of[c, e_], minus], 1)

    tnb = torch.stack([torch.cat([faces_e(owner, e_), faces_e(neigh, e_)], 1) for e_ in range(3)], 1)

    kind = torch.zeros(n, dtype=torch.long)
    for k, cap in enumerate(grid["caps"]):
        s_n = (pos - cap["centre"]) @ cap["normal"]  # signed distance from the cap plane (negative inside)
        lateral = (pos - cap["centre"] - s_n[:, None] * cap["normal"]).norm(dim=1)
        near = (s_n > -1.5 * h) & (lateral < cap["radius"] + 2 * h)
        kind[near] = 1 if k == 0 else 2
    fixed = (kind[owner] == 1) | (kind[neigh] == 1)
    corr = ~fixed & ~((kind[owner] == 2) & (kind[neigh] == 2))
    s_in = torch.where((kind[owner] == 1) & (kind[neigh] != 1), 1.0, 0.0) - torch.where(
        (kind[neigh] == 1) & (kind[owner] != 1), 1.0, 0.0
    )
    s_out = torch.where((kind[neigh] == 2) & (kind[owner] == 0), 1.0, 0.0) - torch.where(
        (kind[owner] == 2) & (kind[neigh] == 0), 1.0, 0.0
    )
    interior = kind == 0
    deg = torch.zeros(n, dtype=torch.float64).index_add(
        0, owner[corr], torch.ones(int(corr.sum()), dtype=torch.float64)
    )
    deg = deg.index_add(0, neigh[corr], torch.ones(int(corr.sum()), dtype=torch.float64))
    outlet_of = torch.full((n,), -1, dtype=torch.long)
    if len(grid["caps"]) > 1:
        centres = torch.stack([c["centre"] for c in grid["caps"][1:]])
        outlet_of = torch.where(kind == 2, torch.cdist(pos, centres).argmin(1), -1)
    return {
        "h": h, "lo": lo, "shape": shape, "idx": idx, "cells": cells, "pos": pos, "nb": nb, "n": n,
        "owner": owner, "neigh": neigh, "dirn": dirn, "fpos": fpos, "face_of": face_of, "fnb": fnb, "ghost": ghost,
        "kind": kind, "fixed": fixed, "corr": corr, "s_in": s_in, "s_out": s_out, "interior": interior,
        "diag": deg / h**2, "periodic": periodic, "phi": phi, "inside": inside, "caps": grid["caps"],
        "outlet_of": outlet_of, "tnb": tnb,
    }  # fmt: skip


# ---------------------------------------------------------------- pressure: matrix assembly and two-level PCG
def pressure_matrix(mesh: dict, block: int = 4) -> dict:
    """Assemble A = −∇² on the interior cells from the face list (OpenFOAM's fvm::laplacian → lduMatrix), as a
    sparse CSR matrix, plus a coarse space: aggregates of block³ voxels with A_c = Pᵀ A P (piecewise constant P)."""
    h, corr, inter = mesh["h"], mesh["corr"], mesh["interior"]
    I = inter.nonzero()[:, 0]  # noqa: E741  (the unknowns: interior cells)
    m = len(I)
    loc = torch.full((mesh["n"],), -1, dtype=torch.long)
    loc[I] = torch.arange(m)
    a_, b_ = loc[mesh["owner"][corr]], loc[mesh["neigh"][corr]]
    diag = torch.zeros(m, dtype=torch.float64)
    rows, cols, vals = [torch.arange(m)], [torch.arange(m)], []
    for a, b in ((a_, b_), (b_, a_)):  # each face couples its two cells symmetrically
        ok = a >= 0
        diag.index_add_(0, a[ok], torch.full((int(ok.sum()),), 1 / h**2, dtype=torch.float64))
        both = ok & (b >= 0)  # a neighbour in an outlet cell has p = 0 and only adds to the diagonal
        (
            rows.append(a[both]),
            cols.append(b[both]),
            vals.append(torch.full((int(both.sum()),), -1 / h**2, dtype=torch.float64)),
        )
    vals.insert(0, diag)
    A = torch.sparse_coo_tensor(torch.stack([torch.cat(rows), torch.cat(cols)]), torch.cat(vals), (m, m)).coalesce()
    c = mesh["cells"][I] // block
    _, agg = torch.unique(c[:, 0] * 10**8 + c[:, 1] * 10**4 + c[:, 2], return_inverse=True)
    na = int(agg.max()) + 1
    # A_c[I, J] = Σ_{a ∈ I, b ∈ J} A_ab: sum the matrix entries by aggregate pairs
    ij = A.indices()
    Ac = (
        torch.zeros(na * na, dtype=torch.float64)
        .index_add_(0, agg[ij[0]] * na + agg[ij[1]], A.values())
        .reshape(na, na)
    )
    return {"A": A.to_sparse_csr(), "I": I, "agg": agg, "Aci": torch.linalg.pinv(Ac), "dinv": 1 / diag, "m": m}


def pcg(A, b, x, M, tol: float, maxit: int = 1000, atol: float = 0.0):
    """Preconditioned conjugate gradients for A x = b (A symmetric positive (semi-)definite)."""
    r = b - A @ x
    z = M(r)
    p, rz = z.clone(), r @ z
    stop = max(tol * b.norm().item(), atol)  # ‖r‖ ≤ max(tol ‖b‖, atol)
    k = 0
    for k in range(maxit):  # noqa: B007  (k is returned)
        if r.norm().item() <= stop:
            break
        Ap = A @ p
        alpha = rz / (p @ Ap)
        x, r = x + alpha * p, r - alpha * Ap
        z = M(r)
        rz, rz_old = r @ z, rz
        p = z + (rz / rz_old) * p
    return x, k


def project(u: torch.Tensor, mesh: dict, dt: float, p0: torch.Tensor, tol: float = 1e-8):
    """Chorin's projection: solve −∇²p = −∇·u*/Δt on the interior cells (p = 0 in outlet cells, ∂p/∂n = 0 on walls
    and the inlet) and correct u = u* − Δt∇p on the correctable faces. Returns (u, p, PCG iterations).

    Preconditioner: Jacobi plus a coarse-grid correction on the voxel aggregates (two-level additive Schwarz),
    M⁻¹r = D⁻¹r + P A_c⁻¹ Pᵀ r, which cuts the iterations about tenfold.
    """
    if "pm" not in mesh:
        mesh["pm"] = pressure_matrix(mesh)
    pm = mesh["pm"]
    dt_ = u.dtype
    if pm.get("dtype") != dt_:
        pm.update(Ad=pm["A"].to(dt_), Acid=pm["Aci"].to(dt_), dinvd=pm["dinv"].to(dt_), dtype=dt_)
    agg, na = pm["agg"], pm["Aci"].shape[0]

    def M(r):
        return pm["dinvd"] * r + (pm["Acid"] @ torch.zeros(na, dtype=dt_).index_add_(0, agg, r))[agg]

    o, nn, corr = mesh["owner"], mesh["neigh"], mesh["corr"]
    b = (-eq.divergence(u, o, nn, mesh["n"], mesh["h"]) / dt)[pm["I"]]  # right-hand side −∇·u*/Δt on the unknowns
    atol = 1e-3 * tol * u.norm().item() / (dt * mesh["h"])  # floor for b ≈ 0 (periodic pipe: div u* is round-off)
    x, iters = pcg(pm["Ad"], b, p0[pm["I"]].to(dt_), M, tol, atol=atol)  # A = −∇²: A p = −∇·u*/Δt ⇔ ∇²p = ∇·u*/Δt
    p = torch.zeros(mesh["n"], dtype=dt_)
    p[pm["I"]] = x
    u = u.clone()
    u[corr] = u[corr] - dt * eq.face_gradient(p, o[corr], nn[corr], mesh["h"])
    return u, p, iters


# ---------------------------------------------------------------- semi-Lagrangian advection on the staggered grid
def sample(field: torch.Tensor, X: torch.Tensor, lo: torch.Tensor, h: float, offset, border: bool) -> torch.Tensor:
    """Trilinear interpolation of dense fields (C, nx, ny, nz) at points X (M, 3); `offset` (3,) shifts the sample
    positions (½ for a staggered component). → (M, C)."""
    shape = torch.tensor(field.shape[1:], dtype=X.dtype)
    g = (X - lo) / h - torch.as_tensor(offset, dtype=X.dtype)
    g = 2 * g / (shape - 1) - 1  # normalised to [−1, 1]
    grid = (
        g.flip(-1).reshape(1, 1, 1, -1, 3).to(field.dtype)
    )  # grid_sample wants (z, y, x) order for (D, H, W) = (x, y, z)
    pad = "border" if border else "zeros"
    out = Fn.grid_sample(field[None], grid, mode="bilinear", padding_mode=pad, align_corners=True)
    return out[0, :, 0, 0].T  # (M, C)


def dense_faces(u: torch.Tensor, mesh: dict) -> torch.Tensor:
    """Scatter face velocities to three dense staggered grids (3, nx, ny, nz); walls and solid stay 0."""
    D = torch.zeros((3, *mesh["shape"]), dtype=u.dtype)
    c = mesh["cells"][mesh["owner"]]
    D[mesh["dirn"], c[:, 0], c[:, 1], c[:, 2]] = u
    return D


def advect(u: torch.Tensor, mesh: dict, dt: float) -> torch.Tensor:
    """u_new(x_f) = u(x_f − Δt v(x_f)): trace each face centre back along the flow and interpolate there."""
    D, lo, h, border = dense_faces(u, mesh), mesh["lo"], mesh["h"], any(mesh["periodic"])
    e = torch.eye(3, dtype=torch.float64)
    uu = torch.cat([u, u.new_zeros(1)])  # index −1 (a wall face) → 0
    v = uu[mesh["tnb"]].mean(-1)  # (F, 3): other components = mean of the 4 surrounding faces
    v[torch.arange(len(u)), mesh["dirn"]] = u
    X = mesh["fpos"] - dt * v.double()
    out = torch.empty_like(u)
    for c in range(3):
        m = mesh["dirn"] == c
        out[m] = sample(D[c : c + 1], X[m], lo, h, 0.5 * e[c], border)[:, 0]
    return out


# ---------------------------------------------------------------- cell-centred velocity, wall shear stress
def cell_velocity(u: torch.Tensor, mesh: dict) -> torch.Tensor:
    """Dense cell-centred velocity (3, nx, ny, nz): mean of the two faces per direction, extended into the first
    solid layers by linear extrapolation through u = 0 on the wall (u_s = u_f φ_s/φ_f), for wall sampling."""
    fo, nb = mesh["face_of"], mesh["nb"]
    uu = torch.cat([u, u.new_zeros(1)])  # index −1 → 0 (a wall face)
    Uc = torch.stack(
        [
            0.5 * (uu[fo[:, d]] + uu[torch.where(nb[:, 2 * d] >= 0, fo[nb[:, 2 * d].clamp(min=0), d], -1)])
            for d in range(3)
        ]
    )  # (3, N)
    dense = torch.zeros((3, *mesh["shape"]), dtype=u.dtype)
    c = mesh["cells"]
    dense[:, c[:, 0], c[:, 1], c[:, 2]] = Uc
    return extend_into_wall(dense, mesh)


def extend_into_wall(dense: torch.Tensor, mesh: dict) -> torch.Tensor:
    """Fill the first solid layers of a dense cell-centred field (3, nx, ny, nz) by linear extrapolation through
    u = 0 on the wall, u_s = φ_s · mean(u_f/φ_f) over the fluid neighbours."""
    phi, inside, h = mesh["phi"].to(dense.dtype), mesh["inside"], mesh["h"]
    w = torch.where(inside, 1 / phi.clamp(min=0.1 * h), 0.0)  # weight u_f/φ_f from fluid neighbours
    num, cnt = torch.zeros_like(dense), torch.zeros_like(phi)
    for a, s in DIRS:
        num += shift(dense * w, a + 1, s, mesh["periodic"][a], 0.0)
        cnt += shift(inside.to(dense.dtype), a, s, mesh["periodic"][a], 0.0)
    ext = phi.clamp(min=-3 * h) * num / cnt.clamp(min=1)
    return torch.where(inside, dense, torch.where(cnt > 0, ext, 0.0))


def streamlines(Uc: torch.Tensor, mesh: dict, seeds: torch.Tensor, n_steps: int = 150, ds: float | None = None):
    """Batched RK4 streamlines of the frozen cell-centred field (3, nx, ny, nz) from seeds (S, 3), parametrised
    by arc length (step ds along u/|u|). Returns points (S, L, 3) with NaN after a line leaves the fluid, and the
    speed |u| (S, L)."""
    lo, h = mesh["lo"], mesh["h"]
    ds = ds or 0.8 * h
    inside = mesh["inside"].to(Uc.dtype)[None]

    def f(x):
        v = sample(Uc, x, lo, h, (0, 0, 0), False).double()
        return v / v.norm(dim=1, keepdim=True).clamp(min=1e-9)

    x = seeds.clone()
    alive = torch.ones(len(x), dtype=torch.bool)
    X, S = [x.clone()], [sample(Uc, x, lo, h, (0, 0, 0), False).double().norm(dim=1)]
    for _ in range(n_steps):
        k1 = f(x)
        k2 = f(x + 0.5 * ds * k1)
        k3 = f(x + 0.5 * ds * k2)
        k4 = f(x + ds * k3)
        x = x + ds / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        alive &= sample(inside, x, lo, h, (0, 0, 0), False)[:, 0] > 0.5
        X.append(torch.where(alive[:, None], x, float("nan")))
        S.append(sample(Uc, x, lo, h, (0, 0, 0), False).double().norm(dim=1))
    return torch.stack(X, 1), torch.stack(S, 1)


def wss_at(Uc: torch.Tensor, V: torch.Tensor, N: torch.Tensor, mesh: dict, mu: float) -> torch.Tensor:
    """Wall shear stress vectors (P, 3) at wall points V with outward normals N, from samples at h and 2h inside."""
    m, h, lo, border = -N, mesh["h"], mesh["lo"], any(mesh["periodic"])
    u1 = sample(Uc, V + h * m, lo, h, (0, 0, 0), border).double()
    u2 = sample(Uc, V + 2 * h * m, lo, h, (0, 0, 0), border).double()
    return eq.wall_shear(u1, u2, m, h, mu)


# ---------------------------------------------------------------- one time step and the pulsatile run
def inlet_setup(mesh: dict, Qhat: torch.Tensor, omega: float, nu: float) -> dict:
    """Womersley shape per harmonic on the prescribed inlet faces, normalised so that the discrete inflow
    Σ_f s_f u_f h² equals Q(t) exactly. Faces: (Fi,), shapes: (H+1, Fi) complex."""
    cap = mesh["caps"][0]
    f = mesh["fixed"].nonzero()[:, 0]
    rel = mesh["fpos"][f] - cap["centre"]
    r = (rel - (rel @ cap["normal"])[:, None] * cap["normal"]).norm(dim=1)
    comp = -cap["normal"][mesh["dirn"][f]]  # inflow direction component along each face normal
    S = eq.womersley_shapes(r, len(Qhat) - 1, omega, cap["radius"], nu) * comp  # (H+1, Fi)
    flux = (S * mesh["s_in"][f]).sum(1) * mesh["h"] ** 2  # discrete flux of each unit shape
    return {"faces": f, "S": S / flux[:, None], "Qhat": Qhat, "omega": omega}


def inlet_values(inl: dict, t: float) -> torch.Tensor:
    n = torch.arange(len(inl["Qhat"]), dtype=torch.float64)
    phase = torch.exp(1j * inl["omega"] * t * n)
    return torch.einsum("h,h,hf->f", phase, inl["Qhat"].to(torch.complex128), inl["S"]).real


def step(u, p, mesh, dt, nu, t, inl=None, body=None, tol=1e-8):
    r"""u* = advect(u) + Δt(ν∇²u + g); inlet faces prescribed; then project. Returns (u, p, CG iterations)."""
    us = advect(u, mesh, dt) + dt * nu * eq.face_laplacian(u, mesh["fnb"], mesh["ghost"].to(u.dtype), mesh["h"])
    if body is not None:
        us = us + dt * body(t)[mesh["dirn"]].to(u.dtype)
    if inl is not None:
        us[inl["faces"]] = inlet_values(inl, t + dt).to(u.dtype)
    return project(us, mesh, dt, p, tol)


def fourier(Q: torch.Tensor, n_harm: int) -> torch.Tensor:
    """Q(t) = Re Σ_n Q̂_n e^{inωt} from K equally spaced samples over one period (Q̂_0 = mean)."""
    X = torch.fft.fft(Q.to(torch.complex128)) / len(Q)
    X[1:] *= 2
    return X[: n_harm + 1]


def flows(u: torch.Tensor, mesh: dict) -> tuple[float, torch.Tensor]:
    """Inflow Σ s_in u h² and the outflow through each outlet cap (m³/s)."""
    h2 = mesh["h"] ** 2
    q_in = (mesh["s_in"] * u).sum().item() * h2
    which = mesh["outlet_of"][torch.where(mesh["kind"][mesh["neigh"]] == 2, mesh["neigh"], mesh["owner"])]
    q_out = torch.zeros(len(mesh["caps"]) - 1, dtype=torch.float64).index_add(
        0, which.clamp(min=0), (mesh["s_out"] * u).double() * h2
    )
    return q_in, q_out


def simulate(mesh, waveform, cycles, dt, nu, rho, V, N, n_frames=48, n_wss=50, n_particles=2000, slice_axis=1,
             slice_at=None, seed=0, dtype=torch.float32, probe=None, n_stream=60):  # fmt: skip
    """Pulsatile run. Records (in the last cycle) WSS at the wall vertices, a velocity-magnitude slice, tracer
    particles, streamlines, inflow/outflow and CG iterations; with `probe` = {"times": absolute times, "points":
    (S, 3) in m} also |u| at those points (trilinear) and |WSS| at V at those times (OpenFOAM comparison)."""
    t_wave, Q = waveform
    T = float(t_wave[-1] + (t_wave[1] - t_wave[0]))
    omega = 2 * math.pi / T
    inl = inlet_setup(mesh, fourier(Q, 12), omega, nu)
    n_steps = int(round(cycles * T / dt))
    last = n_steps - int(round(T / dt))
    frame_steps = set(last + round(k * (n_steps - last - 1) / (n_frames - 1)) for k in range(n_frames))
    wss_steps = set(last + round(k * (n_steps - last - 1) / (n_wss - 1)) for k in range(n_wss))
    u = torch.zeros(len(mesh["owner"]), dtype=dtype)
    u[inl["faces"]] = inlet_values(inl, 0.0).to(dtype)
    p = torch.zeros(mesh["n"], dtype=dtype)
    gen = torch.Generator().manual_seed(seed)
    cap = mesh["caps"][0]
    ex = torch.linalg.cross(cap["normal"], torch.tensor([0.3, 0.5, 0.8], dtype=torch.float64))
    ex = ex / ex.norm()
    ey = torch.linalg.cross(cap["normal"], ex)

    def spawn(k):  # k points on the inlet disc, 2–12 mm downstream of the cap plane
        rr = 0.8 * cap["radius"] * torch.sqrt(torch.rand(k, generator=gen, dtype=torch.float64))
        th = 2 * math.pi * torch.rand(k, generator=gen, dtype=torch.float64)
        depth = 2e-3 + 10e-3 * torch.rand(k, generator=gen, dtype=torch.float64)
        return (
            cap["centre"]
            - depth[:, None] * cap["normal"]
            + rr[:, None] * (torch.cos(th)[:, None] * ex + torch.sin(th)[:, None] * ey)
        )

    P = spawn(n_particles)
    age = torch.zeros(n_particles, dtype=torch.float64)  # time since each tracer entered (s)
    inside_f = mesh["inside"].to(dtype)[None]
    rec = {
        "t": [],
        "Q_in": [],
        "Q_out": [],
        "wss": [],
        "wss_t": [],
        "slice": [],
        "P": [],
        "iters": [],
        "dp": [],
        "stream": [],
        "stream_speed": [],
        "probe_u": [],
        "probe_wss": [],
        "probe_t": [],
        "P_age": [],
        "P_speed": [],
    }
    gs = torch.Generator().manual_seed(seed + 1)
    rr = 0.85 * cap["radius"] * torch.sqrt(torch.rand(n_stream, generator=gs, dtype=torch.float64))
    th = 2 * math.pi * torch.rand(n_stream, generator=gs, dtype=torch.float64)
    stream_seeds = (
        cap["centre"]
        - 3 * mesh["h"] * cap["normal"]
        + rr[:, None] * (torch.cos(th)[:, None] * ex + torch.sin(th)[:, None] * ey)
    )
    s_in = (mesh["pos"] - cap["centre"]) @ cap["normal"]
    near_in = (mesh["kind"] == 0) & (s_in > -4 * mesh["h"])  # first interior layers after the inlet
    sidx = slice_at if slice_at is not None else mesh["shape"][slice_axis] // 2
    t = 0.0
    for k in steps(n_steps, "time steps"):
        u, p, it = step(u, p, mesh, dt, nu, t, inl=inl, tol=1e-4 if dtype == torch.float32 else 1e-9)
        t += dt
        rec["iters"].append(it)
        if k in wss_steps or k in frame_steps:
            Ucp = cell_velocity(u, mesh)
        # tracers: forward Euler with the staggered velocity; respawn when they leave the fluid or reach an outlet
        D = dense_faces(u, mesh)
        e = torch.eye(3, dtype=torch.float64)
        vP = torch.cat([sample(D[c : c + 1], P, mesh["lo"], mesh["h"], 0.5 * e[c], False) for c in range(3)], 1)
        P = P + dt * vP.double()
        age += dt
        out = sample(inside_f, P, mesh["lo"], mesh["h"], (0, 0, 0), False)[:, 0] < 0.5
        for c in mesh["caps"][1:]:
            out |= (P - c["centre"]) @ c["normal"] > -mesh["h"]
        if out.any():
            P[out] = spawn(int(out.sum()))
            age[out] = 0.0
        if k in wss_steps:
            rec["wss"].append(wss_at(Ucp, V, N, mesh, rho * nu).float())
            rec["wss_t"].append(t)
        if probe is not None and (probe["times"] - t).abs().min() < 0.5 * dt:
            Upr = cell_velocity(u, mesh)
            rec["probe_u"].append(
                sample(Upr, probe["points"], mesh["lo"], mesh["h"], (0, 0, 0), False).norm(dim=1).double()
            )
            rec["probe_wss"].append(wss_at(Upr, V, N, mesh, rho * nu).norm(dim=1))
            rec["probe_t"].append(t)
        if k in frame_steps:
            sl_x, sl_s = streamlines(Ucp, mesh, stream_seeds)
            rec["stream"].append(sl_x.float()), rec["stream_speed"].append(sl_s.float())
            q_in, q_out = flows(u, mesh)
            rec["t"].append(t), rec["Q_in"].append(q_in), rec["Q_out"].append(q_out)
            rec["slice"].append(Ucp.norm(dim=0).select(slice_axis, sidx).clone())
            rec["P"].append(P.clone())
            rec["P_age"].append(age.clone()), rec["P_speed"].append(vP.double().norm(dim=1))
            rec["dp"].append(rho * p[near_in].mean().item())  # inlet-to-outlet pressure drop (outlets: p = 0)
    rec["p"] = p
    rec["u"] = u
    rec["T"] = T
    rec["wss"] = torch.stack(rec["wss"]).double()
    rec["slice"] = torch.stack(rec["slice"])
    rec["P"] = torch.stack(rec["P"])
    rec["Q_out"] = torch.stack(rec["Q_out"])
    rec["slice_index"] = sidx
    rec["stream"], rec["stream_speed"] = torch.stack(rec["stream"]), torch.stack(rec["stream_speed"])
    rec["P_age"], rec["P_speed"] = torch.stack(rec["P_age"]), torch.stack(rec["P_speed"])
    if rec["probe_u"]:
        rec["probe_u"], rec["probe_wss"] = torch.stack(rec["probe_u"]), torch.stack(rec["probe_wss"]).double()
    return rec


def pipe_run(grid, mesh, G0, G1, omega, t_end, dt, nu, rho=1060.0, n_rec=None):
    """Periodic pipe driven by −∂_z p/ρ = G0 + G1 sin(ωt): axis velocity, half-radius velocity, mean WSS over time."""
    body = lambda t: torch.tensor([0.0, 0.0, G0 + G1 * math.sin(omega * t)], dtype=torch.float64)  # noqa: E731
    u = torch.zeros(len(mesh["owner"]), dtype=torch.float64)
    p = torch.zeros(mesh["n"], dtype=torch.float64)
    n_steps = int(round(t_end / dt))
    every = 1 if n_rec is None else max(1, n_steps // n_rec)
    r = (mesh["pos"][:, :2]).norm(dim=1)
    axis = r < 1e-12
    half = (r - 0.5 * grid["phi"].max().item()).abs() < 0.5 * mesh["h"]  # cells near r = R/2
    out = {"t": [], "u_axis": [], "u_half": [], "wss": [], "Q": []}
    t = 0.0
    for k in range(n_steps):
        u, p, _ = step(u, p, mesh, dt, nu, t, body=body)
        t += dt
        if (k + 1) % every == 0:
            Uc = cell_velocity(u, mesh)
            uz = Uc[2][tuple(mesh["cells"].T)]
            tau = wss_at(Uc, grid["V"], grid["N"], mesh, rho * nu)
            (
                out["t"].append(t),
                out["u_axis"].append(uz[axis].mean().item()),
                out["u_half"].append(uz[half].mean().item()),
            )
            out["wss"].append(tau[:, 2].mean().item())
            out["Q"].append((u * (mesh["dirn"] == 2)).sum().item() * mesh["h"] ** 2 / mesh["shape"][2])
    out = {k: torch.tensor(v, dtype=torch.float64) for k, v in out.items()}
    out["u"] = u
    return out


# ---------------------------------------------------------------- centreline (tensor version of vmtkcenterlines)
def medial_graph(grid: dict, power: float = 2.0):
    """Fluid cells as graph nodes, 26-neighbour edges with eikonal travel times ℓ/R^p (R = φ, the inscribed radius)."""
    inside, h = grid["inside"], grid["h"]
    shape = inside.shape
    n = int(inside.sum())
    idx = torch.full(shape, -1, dtype=torch.long)
    idx[inside] = torch.arange(n)
    R = grid["phi"][inside].clamp(min=0.05 * h)  # (N,)
    src, dst, w = [], [], []
    for off in torch.cartesian_prod(*[torch.tensor([-1, 0, 1])] * 3):
        if (off == 0).all():
            continue
        nbr = idx
        for a in range(3):
            if off[a] != 0:
                nbr = shift(nbr, a, int(off[a]), False, -1)
        j = nbr[inside]
        ok = j >= 0
        i = torch.arange(n)[ok]
        src.append(i), dst.append(j[ok])
        w.append(
            eq.edge_cost(
                h * off.double().norm() * torch.ones(len(i), dtype=torch.float64), 0.5 * (R[i] + R[j[ok]]), power
            )
        )
    pos = grid["lo"] + h * inside.nonzero().double()
    return {"src": torch.cat(src), "dst": torch.cat(dst), "w": torch.cat(w), "R": R, "pos": pos, "n": n}


def eikonal(graph: dict, sources: torch.Tensor, max_iter: int = 5000):
    """Arrival times T from the source nodes by iterated min-plus relaxation (a tensor stand-in for fast marching),
    and the predecessor of every node on its shortest path."""
    T = torch.full((graph["n"],), float("inf"), dtype=torch.float64)
    T[sources] = 0.0
    for it in range(max_iter):  # noqa: B007  (it is returned)
        Tn = eq.relax(T, graph["src"], graph["dst"], graph["w"])
        if torch.equal(Tn, T):
            break
        T = Tn
    cand = T[graph["src"]] + graph["w"]
    best = (cand <= T[graph["dst"]] * (1 + 1e-12)) & torch.isfinite(cand)
    pred = torch.full((graph["n"],), -1, dtype=torch.long)
    pred[graph["dst"][best]] = graph["src"][best]
    pred[sources] = -1
    return T, pred, it


def backtrack(pred: torch.Tensor, seeds: torch.Tensor, max_len: int = 100000) -> list:
    """Steepest descent of T from all outlet seeds at once (follow the predecessors to the inlet)."""
    cur = seeds.clone()
    paths = [cur.clone()]
    for _ in range(max_len):
        nxt = torch.where(cur >= 0, pred[cur.clamp(min=0)], -1)
        if (nxt < 0).all():
            break
        cur = nxt
        paths.append(cur.clone())
    P = torch.stack(paths, 1)  # (K, L) node ids, −1 after reaching the inlet
    return [row[row >= 0].flip(0) for row in P]  # inlet → outlet


def smooth(x: torch.Tensor, half: int) -> torch.Tensor:
    """Moving average along a polyline (L, C), ends kept by edge padding."""
    if len(x) < 3:
        return x
    k = torch.ones(1, 1, 2 * half + 1, dtype=x.dtype) / (2 * half + 1)
    xp = Fn.pad(x.T[None], (half, half), mode="replicate")
    return Fn.conv1d(xp.reshape(-1, 1, xp.shape[-1]), k).reshape(x.shape[1], -1).T


def centrelines(grid: dict, inlet_nodes: torch.Tensor, outlet_seeds: torch.Tensor, power: float = 2.0, half: int = 3):
    """Centrelines from the inlet to every outlet: positions (L, 3), radius (L,), abscissa (L,) per line, and the
    bifurcation point where the first two lines separate."""
    g = medial_graph(grid, power)
    T, pred, iters = eikonal(g, inlet_nodes)
    lines = []
    for path in backtrack(pred, outlet_seeds):
        x = smooth(g["pos"][path], half)
        R = smooth(g["R"][path][:, None], half)[:, 0]
        s = torch.cat([x.new_zeros(1), (x[1:] - x[:-1]).norm(dim=1).cumsum(0)])
        lines.append({"nodes": path, "x": x, "R": R, "s": s})
    bif = None
    if len(lines) >= 2:
        a, b = lines[0]["nodes"], lines[1]["nodes"]
        m = min(len(a), len(b))
        same = (a[:m] == b[:m]).long()
        j = int(same.cumprod(0).sum()) - 1  # last shared node
        bif = {"x": lines[0]["x"][max(j, 0)], "s": lines[0]["s"][max(j, 0)].item()}
    return lines, bif, {"iters": iters, "graph": g}


def cap_nodes(grid: dict, cap: dict) -> torch.Tensor:
    """Graph nodes in the slab next to a cap (within 1.5 voxels of its plane)."""
    pos = grid["lo"] + grid["h"] * grid["inside"].nonzero().double()
    s = (pos - cap["centre"]) @ cap["normal"]
    lat = (pos - cap["centre"] - s[:, None] * cap["normal"]).norm(dim=1)
    return ((s > -1.5 * grid["h"]) & (lat < cap["radius"])).nonzero()[:, 0]


def outlet_seed(grid: dict, cap: dict) -> int:
    """The most central node of an outlet slab (largest inscribed radius)."""
    nodes = cap_nodes(grid, cap)
    return int(nodes[grid["phi"][grid["inside"]][nodes].argmax()])


# ---------------------------------------------------------------- residuals against the precomputed references
def harmonic(y: torch.Tensor, t: torch.Tensor, omega: float) -> complex:
    """First Fourier coefficient c with y ≈ mean + Re(c e^{iωt}) over whole periods."""
    return complex((y * torch.exp(-1j * omega * t)).mean().item()) * 2


def openfoam_residuals(ours: dict, ref: dict, t0: float, t1: float, rho: float) -> dict:
    """Our pipe run minus OpenFOAM's on the OpenFOAM sample times in (t0, t1]: relative L2 and L∞ errors of the
    axis velocity, WSS and flow rate, and the amplitude/phase error of the axis velocity's first harmonic."""
    sel = (ref["t"] > t0 + 1e-9) & (ref["t"] <= t1 + 1e-9)
    t = ref["t"][sel]
    k = torch.searchsorted(ours["t"], t - 1e-9)
    out = {"t": t}
    for name, scale in (("u_axis", 1.0), ("wss", 1 / rho), ("Q", 1.0)):
        a, b = ours[name][k] * scale, ref[name][sel]
        out[name] = (a, b)
        out[name + "_L2"] = ((a - b).norm() / b.norm()).item()
        out[name + "_Linf"] = ((a - b).abs().max() / b.abs().max()).item()
    omega = 2 * math.pi / (t1 - t0)
    ca, cb = harmonic(out["u_axis"][0], t, omega), harmonic(out["u_axis"][1], t, omega)
    out["amp_err"] = abs(ca) / abs(cb) - 1
    out["phase_err"] = math.remainder(math.atan2(ca.imag, ca.real) - math.atan2(cb.imag, cb.real), 2 * math.pi)
    return out


def vmtk_residuals(lines: list, P: torch.Tensor, R: torch.Tensor, lid: torch.Tensor) -> list:
    """For each VMTK line: distance of its points to our polyline and |ΔR| at the nearest of our points (m)."""
    res = []
    for k in lid.unique().tolist():  # VMTK lines that exist (a degenerate one may have been left out)
        line = lines[k]
        ref, Rr = P[lid == k], R[lid == k]
        D = torch.cdist(ref, line["x"])
        d, j = D.min(1)
        dr = (line["R"][j] - Rr).abs()
        s_ref = torch.cat([ref.new_zeros(1), (ref[1:] - ref[:-1]).norm(dim=1).cumsum(0)])
        res.append({"k": k, "d": d, "dR": dr, "s_ref": s_ref, "R_ref": Rr})
    return res


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/aneurysm_cfd/plot.py
"""Eight panels over the last cardiac cycle: wall shear stress on the surface; streamlines and tracers inside the
semi-transparent vessel; a velocity slice; ours − OpenFOAM on that slice; the inflow waveform; ours − OpenFOAM TAWSS
on the wall; the pipe check against OpenFOAM; the centreline check against VMTK. The last frames show TAWSS and OSI.
Also writes coloured surfaces as binary PLY files for the interactive 3D view."""

import base64
import xml.etree.ElementTree as ET
import zlib
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
from PIL import Image

from problems._common.plotting import export_plot_data
from problems._common.progress import report

N_SUMMARY = 4  # frames that hold the TAWSS / OSI maps at the end
MM = 1e3


def _box(ax, P):
    c, r = (P.max(0) + P.min(0)) / 2, (P.max(0) - P.min(0)).max() / 2
    ax.set_xlim(c[0] - r, c[0] + r), ax.set_ylim(c[1] - r, c[1] + r), ax.set_zlim(c[2] - r, c[2] + r)
    for lab, f in (("$x$ (mm)", ax.set_xlabel), ("$y$ (mm)", ax.set_ylabel), ("$z$ (mm)", ax.set_zlabel)):
        f(lab, fontsize=7, labelpad=-8)
    ax.tick_params(labelsize=6, pad=-3)
    ax.view_init(elev=14, azim=-62)
    ax.set_box_aspect(None, zoom=1.15)


def wall_mesh(tris: torch.Tensor, ids: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    """Wall vertices (V, 3) in the same order as tensors.vertices_and_normals, and triangles (F, 3) as indices."""
    wall = tris[ids == 0]
    V, inv = torch.unique((wall.reshape(-1, 3) / 1e-9).round(), dim=0, return_inverse=True)
    return (V * 1e-9).numpy(), inv.reshape(-1, 3).numpy()


def display_mesh(tris, ids, grid, cell: float = 0.7e-3):
    """Decimated wall mesh for display: vertices clustered on a `cell` grid. Returns cluster points (C, 3) in mm,
    triangles (F, 3) and a function mapping per-vertex values (…, V) to per-cluster values (…, C)."""
    _, tri_v = wall_mesh(tris, ids)
    _, cl = torch.unique((grid["V"] / cell).round().long(), dim=0, return_inverse=True)
    cl = cl.numpy()
    nc = cl.max() + 1
    cnt = np.bincount(cl, minlength=nc)
    Vc = np.stack([np.bincount(cl, grid["V"][:, i].numpy(), nc) / cnt for i in range(3)], 1) * MM
    ct = cl[tri_v]
    ct = ct[(ct[:, 0] != ct[:, 1]) & (ct[:, 1] != ct[:, 2]) & (ct[:, 0] != ct[:, 2])]
    return Vc, ct, lambda a: np.stack([np.bincount(cl, row, nc) / cnt for row in np.atleast_2d(a)])


def _b64(a: np.ndarray) -> str:
    """VTK binary DataArray payload: one zlib block, header [1, n, n, compressed size] (UInt32), both base64."""
    raw = np.ascontiguousarray(a).tobytes()
    c = zlib.compress(raw, 6)
    head = np.array([1, len(raw), len(raw), len(c)], dtype="<u4").tobytes()
    return base64.b64encode(head).decode() + base64.b64encode(c).decode()


def write_vtp(path: Path, points, polys=None, lines=None, verts=False, point_data=None) -> Path:
    """Minimal VTK XML PolyData writer (no vtk dependency): Float32 points, Int32 connectivity/offsets, named point
    arrays {name: (values, unit)}. polys: (M, 3) triangles; lines: list of index arrays; verts: one vertex per point."""
    pts = np.asarray(points, dtype="<f4").reshape(-1, 3)

    def arr(a, name, dtype, ncomp=1, unit=None):
        u = f' unit="{unit}"' if unit else ""
        return (f'<DataArray type="{dtype}" Name="{name}" NumberOfComponents="{ncomp}" format="binary"{u}>'
                f"{_b64(np.asarray(a, dtype='<f4' if dtype == 'Float32' else '<i4'))}</DataArray>")  # fmt: skip

    def cells(conn, offs):
        return arr(conn, "connectivity", "Int32") + arr(offs, "offsets", "Int32")

    counts = {"Verts": 0, "Lines": 0, "Polys": 0}
    body = ""
    if verts:
        counts["Verts"] = len(pts)
        body += "<Verts>" + cells(np.arange(len(pts)), np.arange(1, len(pts) + 1)) + "</Verts>"
    if lines:
        conn = np.concatenate(lines) if lines else np.zeros(0)
        counts["Lines"] = len(lines)
        body += "<Lines>" + cells(conn, np.cumsum([len(x) for x in lines])) + "</Lines>"
    if polys is not None and len(polys):
        counts["Polys"] = len(polys)
        body += "<Polys>" + cells(np.asarray(polys).ravel(), 3 * np.arange(1, len(polys) + 1)) + "</Polys>"
    pd = "".join(arr(v, k, "Float32", unit=u) for k, (v, u) in (point_data or {}).items())
    xml = (
        '<?xml version="1.0"?>\n<VTKFile type="PolyData" version="1.0" byte_order="LittleEndian" header_type="UInt32"'
        ' compressor="vtkZLibDataCompressor"><PolyData>'
        f'<Piece NumberOfPoints="{len(pts)}" NumberOfVerts="{counts["Verts"]}" NumberOfLines="{counts["Lines"]}"'
        f' NumberOfStrips="0" NumberOfPolys="{counts["Polys"]}"><PointData>{pd}</PointData>'
        f"<Points>{arr(pts, 'Points', 'Float32', 3, 'mm')}</Points>{body}</Piece></PolyData></VTKFile>\n"
    )
    Path(path).write_text(xml, encoding="ascii")
    return Path(path)


def read_vtp(path: Path) -> dict:
    """Parse a VTP written by write_vtp back: counts from the Piece and every DataArray decoded (zlib + base64)."""
    root = ET.parse(path).getroot()
    piece = root.find("PolyData/Piece")
    out = {k: int(piece.get(k)) for k in ("NumberOfPoints", "NumberOfVerts", "NumberOfLines", "NumberOfPolys")}
    for da in root.iter("DataArray"):
        text = da.text.strip()
        head = np.frombuffer(base64.b64decode(text[:24]), dtype="<u4")  # 16 bytes → 24 base64 characters
        raw = zlib.decompress(base64.b64decode(text[24:]))
        assert len(raw) == head[1]
        dt = "<f4" if da.get("type") == "Float32" else "<i4"
        a = np.frombuffer(raw, dtype=dt)
        parent = [e.tag for e in root.iter() if da in list(e)][0]
        out[f"{parent}/{da.get('Name')}"] = a.reshape(-1, int(da.get("NumberOfComponents", "1")))
    return out


def write_vtk_series(folder: Path, tris, ids, grid, rec, TA, OS) -> Path:
    """ParaView time series of the recorded cycle: per frame a decimated wall surface (WSS, TAWSS, OSI), the
    streamlines (speed) and the tracers (speed, age), listed in aneurysm.pvd with one group per layer."""
    folder.mkdir(parents=True, exist_ok=True)
    Vc, F, per = display_mesh(tris, ids, grid)
    wss = per(rec["wss"].norm(dim=-1).numpy())
    ta, osi = per(TA.numpy())[0], per(OS.numpy())[0]
    wss_t = np.array(rec["wss_t"])
    tf = (np.array(rec["t"]) - 1e-9) % rec["T"]
    X, S = rec["stream"].numpy() * MM, rec["stream_speed"].numpy()
    entries = []
    for k, t in enumerate(tf):
        j = int(np.abs(wss_t - rec["t"][k]).argmin())
        f1 = write_vtp(folder / f"surface_t{k:03d}.vtp", Vc, polys=F,
                       point_data={"WSS": (wss[j], "Pa"), "TAWSS": (ta, "Pa"), "OSI": (osi, "1")})  # fmt: skip
        pts, lines, spd, n0 = [], [], [], 0
        for s in range(X.shape[1]):
            ok = ~np.isnan(X[k, s, :, 0])
            n = int(ok.sum())
            if n >= 2:
                pts.append(X[k, s, ok]), spd.append(S[k, s, ok]), lines.append(np.arange(n0, n0 + n))
                n0 += n
        f2 = write_vtp(folder / f"streamlines_t{k:03d}.vtp", np.concatenate(pts), lines=lines,
                       point_data={"speed": (np.concatenate(spd), "m/s")})  # fmt: skip
        f3 = write_vtp(folder / f"particles_t{k:03d}.vtp", rec["P"][k].numpy() * MM, verts=True,
                       point_data={"speed": (rec["P_speed"][k].numpy(), "m/s"),
                                   "age": (rec["P_age"][k].numpy(), "s")})  # fmt: skip
        for part, (group, f) in enumerate((("surface", f1), ("streamlines", f2), ("particles", f3))):
            entries.append(f'<DataSet timestep="{t:.4f}" group="{group}" part="{part}" file="{f.name}"/>')
    # the openings (STL attribute 1 inlet, 2, 3 outlets): static layers in a solid colour, shown from the first step
    for name, sel, colour in (("inlet", ids == 1, "#16a34a"), ("outlets", ids >= 2, "#dc2626")):
        cap = tris[sel].reshape(-1, 3).numpy() * MM
        f = write_vtp(folder / f"{name}.vtp", cap, polys=np.arange(len(cap)).reshape(-1, 3))
        entries.append(f'<DataSet timestep="{tf[0]:.4f}" group="{name}" part="{3 + (name == "outlets")}" '
                       f'file="{f.name}" color="{colour}"/>')  # fmt: skip
    pvd = folder / "aneurysm.pvd"
    pvd.write_text('<?xml version="1.0"?>\n<VTKFile type="Collection" version="1.0" byte_order="LittleEndian">'
                   "<Collection>\n" + "\n".join(entries) + "\n</Collection></VTKFile>\n", encoding="ascii")  # fmt: skip
    return pvd


def animate(tris, ids, grid, mesh, lines, vres, rec, TA, OS, sac, pres, ares, waveform, path: Path, title: str = ""):
    _, tri_v = wall_mesh(tris, ids)
    wss = rec["wss"].norm(dim=-1).numpy()  # (T, V) |τ| in Pa
    # lighter display mesh: cluster the wall vertices on a 0.7 mm grid (values averaged per cluster)
    _, cl = torch.unique((grid["V"] / 0.7e-3).round().long(), dim=0, return_inverse=True)
    cl = cl.numpy()
    nc = cl.max() + 1
    cnt = np.bincount(cl, minlength=nc)
    Vc = np.stack([np.bincount(cl, grid["V"][:, i].numpy(), nc) / cnt for i in range(3)], 1) * MM
    ct = cl[tri_v]
    tri_v = ct[(ct[:, 0] != ct[:, 1]) & (ct[:, 1] != ct[:, 2]) & (ct[:, 0] != ct[:, 2])]
    polys = Vc[tri_v]
    per_cluster = lambda a: np.stack([np.bincount(cl, row, nc) / cnt for row in np.atleast_2d(a)])  # noqa: E731
    wss = per_cluster(wss)
    TAc, OSc = per_cluster(TA.numpy())[0], per_cluster(OS.numpy())[0]
    face = lambda v: v[tri_v].mean(1)  # noqa: E731  vertex values → triangle values
    wss_t = np.array(rec["wss_t"])
    vmax = float(max(wss.max(), TAc.max()))  # full-data limit, fixed over all frames
    norm1 = matplotlib.colors.PowerNorm(0.5, vmin=0, vmax=vmax)  # square-root scale: low sac WSS stays visible
    cmap = plt.get_cmap("inferno", 12)
    Pall = polys.reshape(-1, 3)

    fig = plt.figure(figsize=(15.6, 7.4))
    ax = [fig.add_subplot(2, 4, k + 1, projection="3d" if k in (0, 1, 5) else None) for k in range(8)]

    # 1. wall shear stress on the surface
    surf = Poly3DCollection(polys, linewidths=0)
    surf.set_facecolor(cmap(norm1(face(wss[0]))))
    ax[0].add_collection3d(surf)
    _box(ax[0], Pall)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm1)
    fig.colorbar(sm, ax=ax[0], shrink=0.6, pad=0.1).set_label(r"$|\boldsymbol{\tau}_w|$ (Pa)")
    t1 = ax[0].set_title("wall shear stress", fontsize=10)

    # 2. streamlines (coloured by speed) and tracers inside the semi-transparent vessel
    shell = Poly3DCollection(polys, linewidths=0, facecolor=(0.6, 0.65, 0.75, 0.07))
    ax[1].add_collection3d(shell)
    X, Sp = rec["stream"].numpy() * MM, rec["stream_speed"].numpy()  # (T, S, L, 3), (T, S, L)
    X, Sp = X[:, :, ::2], Sp[:, :, ::2]
    smax_s = float(np.nanmax(np.where(np.isnan(X[..., 0]), np.nan, Sp)))
    snorm = plt.Normalize(0, smax_s)

    def segs(k):
        a, b = X[k, :, :-1], X[k, :, 1:]
        ok = ~(np.isnan(a[..., 0]) | np.isnan(b[..., 0]))
        return np.stack([a[ok], b[ok]], 1), Sp[k, :, :-1][ok]

    s0, c0 = segs(0)
    stream = Line3DCollection(s0, cmap="viridis", norm=snorm, linewidths=0.9, alpha=0.6)
    stream.set_array(c0)
    ax[1].add_collection3d(stream)
    cb_s = fig.colorbar(stream, ax=ax[1], shrink=0.6, pad=0.1)
    cb_s.set_label(r"streamline speed $|\mathbf{u}|$ (m/s)")
    for ln in lines:
        x = ln["x"].numpy() * MM
        ax[1].plot(x[:, 0], x[:, 1], x[:, 2], color="#111827", lw=0.8, alpha=0.5)
    Pp = rec["P"][:, ::4].numpy() * MM  # every 4th tracer in the GIF (all of them in the VTK files)
    in_sac = _in_sac(rec["P"][:, ::4], mesh, sac)
    part = ax[1].scatter(*Pp[0].T, s=2.5, c=np.where(in_sac[0], "#dc2626", "#1e3a8a"), depthshade=False)
    osi_max = max(0.05, float(OSc.max()))
    osi_sm = plt.cm.ScalarMappable(cmap=plt.get_cmap("plasma", 10), norm=plt.Normalize(0, osi_max))
    osi_surf = Poly3DCollection(polys, linewidths=0, visible=False)
    osi_surf.set_facecolor(plt.get_cmap("plasma", 10)(np.clip(face(OSc) / osi_max, 0, 1)))
    ax[1].add_collection3d(osi_surf)
    _box(ax[1], Pall)
    t2 = ax[1].set_title("streamlines, tracers (red: in the sac)", fontsize=10)

    # 3. velocity-magnitude slice through the sac (x–z plane)
    sl = rec["slice"].numpy()  # (T, nx, nz)
    solid = ~grid["inside"].select(1, rec["slice_index"]).numpy()
    lo, h = grid["lo"].numpy() * MM, grid["h"] * MM
    ext = [lo[0] - h / 2, lo[0] + h * (sl.shape[1] - 0.5), lo[2] - h / 2, lo[2] + h * (sl.shape[2] - 0.5)]
    img = ax[2].imshow(np.ma.masked_array(sl[0], solid).T, origin="lower", extent=ext,
                       cmap=plt.get_cmap("viridis", 12),
                       norm=matplotlib.colors.PowerNorm(0.5, vmin=0, vmax=float(sl[:, ~solid].max())))  # fmt: skip
    for a in (ax[2], ax[3]):
        a.set_facecolor("#f3f4f6"), a.grid(False), a.set_xlabel("$x$ (mm)"), a.set_ylabel("$z$ (mm)")
    fig.colorbar(img, ax=ax[2], shrink=0.85).set_label(r"$|\mathbf{u}|$ (m/s)")
    ax[2].set_title(f"slice $y$ = {lo[1] + h * rec['slice_index']:.1f} mm", fontsize=10)

    # 4. ours − OpenFOAM on the slice (same voxel mesh), fixed symmetric scale
    tf = (np.array(rec["t"]) - 1e-9) % rec["T"]
    if ares is not None:
        du = ares["du"].numpy()  # (S, T)
        pts = ares["ref"]["points"].numpy() * MM  # OpenFOAM slice points (cell centres of its 0.4 mm mesh)
        tref = ares["ref"]["t"].numpy() % rec["T"]
        lim = float(np.abs(du).max())  # full-data symmetric limit
        dimg = ax[3].scatter(pts[:, 0], pts[:, 2], c=du[:, 0], cmap="RdBu_r", vmin=-lim, vmax=lim, marker="s", s=14)
        ax[3].set_xlim(ext[0], ext[1]), ax[3].set_ylim(ext[2], ext[3]), ax[3].set_aspect("equal")
        fig.colorbar(dimg, ax=ax[3], shrink=0.85).set_label(r"$\Delta|\mathbf{u}|$ ours − OpenFOAM (m/s)")
        ax[3].set_title(f"vs OpenFOAM: L2 {ares['u_L2'] * 100:.0f} %, L∞ {ares['u_Linf'] * 100:.0f} %", fontsize=10)
    else:
        dimg = None
        ax[3].text(0.5, 0.5, "no OpenFOAM reference\nfor the synthetic geometry", ha="center",
                   va="center", transform=ax[3].transAxes)  # fmt: skip
        ax[3].set_title("ours − OpenFOAM on the slice", fontsize=10)

    # 5. inflow waveform with a cursor
    t_w, Q = waveform
    T = rec["T"]
    ax[4].plot(np.r_[t_w.numpy(), T], np.r_[Q.numpy(), Q[0].item()] * 1e6, color="#111827", lw=1.5,
               label="prescribed $Q(t)$")  # fmt: skip
    ax[4].plot(tf, np.array(rec["Q_in"]) * 1e6, "o", ms=2.5, color="#2563eb", label="discrete inflow")
    ax[4].plot(tf, rec["Q_out"].sum(1).numpy() * 1e6, "x", ms=3, color="#dc2626", label="Σ outflow")
    cursor = ax[4].axvline(tf[0], color="#f59e0b", lw=2)
    ax[4].set_xlabel("time in cycle $t$ (s)"), ax[4].set_ylabel("flow rate $Q$ (ml/s)"), ax[4].set_xlim(0, T)
    ax[4].legend(loc="upper right", fontsize=7)
    vs = _wall_on_sac(grid, mesh, sac)
    ta, os_ = TA.numpy(), OS.numpy()
    note = f"TAWSS sac {np.median(ta[vs]):.2f} Pa, rest {np.median(ta[~vs]):.2f} Pa\n"
    note += f"OSI sac {np.median(os_[vs]):.3f} (max {os_[vs].max():.2f}), rest {np.median(os_[~vs]):.3f}"
    ax[4].text(0.98, 0.42, note, transform=ax[4].transAxes, ha="right", fontsize=7.5)
    ax[4].set_title(f"inflow (mean {Q.mean() * 1e6:.2f} ml/s)", fontsize=10)

    # 6. ours − OpenFOAM TAWSS on the wall
    if ares is not None:
        dta = per_cluster(ares["dtawss"].numpy())[0]
        lim_w = float(np.abs(dta).max())
        dsurf = Poly3DCollection(polys, linewidths=0)
        dsurf.set_facecolor(plt.get_cmap("RdBu_r")(plt.Normalize(-lim_w, lim_w)(face(dta))))
        ax[5].add_collection3d(dsurf)
        dsm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(-lim_w, lim_w))
        fig.colorbar(dsm, ax=ax[5], shrink=0.6, pad=0.1).set_label(r"$\Delta$TAWSS ours − OpenFOAM (Pa)")
        ax[5].set_title(f"TAWSS vs OpenFOAM: L2 {ares['tawss_L2'] * 100:.0f} %, L∞ {ares['tawss_Linf'] * 100:.0f} %",
                        fontsize=10)  # fmt: skip
    else:
        ax[5].text2D(
            0.5, 0.5, "no OpenFOAM reference\nfor the synthetic geometry", ha="center", transform=ax[5].transAxes
        )
        ax[5].set_title("TAWSS: ours − OpenFOAM", fontsize=10)
    _box(ax[5], Pall)

    # 7. pipe check: ours vs OpenFOAM and Womersley, residual on a fixed symmetric scale
    tp = pres["t"].numpy() - 1.0
    ua, uo = (x.numpy() for x in pres["u_axis"])
    ax[6].plot(tp, uo, color="#111827", lw=2.2, label="OpenFOAM pimpleFoam")
    ax[6].plot(tp, ua, color="#2563eb", lw=1.0, label="ours (tensors)")
    ax[6].plot(tp, pres["exact"].numpy(), color="#16a34a", lw=0.8, ls="--", label="Womersley exact")
    ax[6].set_xlabel("time in cycle $t$ (s)"), ax[6].set_ylabel(r"pipe axis velocity $u_{\mathrm{axis}}$ (m/s)")
    ax[6].set_xlim(0, 1)
    r7 = ax[6].twinx()
    res = (ua - uo) * 1e3
    lim7 = max(1.0, float(np.ceil(np.abs(res).max())))
    r7.scatter(tp, res, c=res, cmap="RdBu_r", vmin=-lim7, vmax=lim7, s=6)
    r7.axhline(0, color="#9ca3af", lw=0.8), r7.set_ylim(-4 * lim7, lim7 * 1.2), r7.grid(False)
    r7.set_ylabel(r"$\Delta u$ ours − OpenFOAM (mm/s)", fontsize=8)
    ax[6].legend(loc="lower left", fontsize=6.5)
    ax[6].set_title(f"pipe: L2 {pres['u_axis_L2'] * 100:.1f} % (u), {pres['wss_L2'] * 100:.1f} % (WSS)", fontsize=10)
    cursor7 = ax[6].axvline(tf[0], color="#f59e0b", lw=1.5)

    # 8. centreline vs VMTK: radius along the lines and the distance between the two centrelines
    hv = grid["h"]
    for r, col in zip(vres, ["#2563eb", "#9333ea", "#16a34a"]):
        ln = lines[r["k"]]
        ax[7].plot(ln["s"].numpy() * MM, ln["R"].numpy() * MM, color=col, lw=1.5, label=f"ours, outlet {r['k'] + 1}")
        ax[7].plot(r["s_ref"].numpy() * MM, r["R_ref"].numpy() * MM, color="#111827", lw=0.8, ls="--",
                   label="VMTK" if r is vres[0] else None)  # fmt: skip
    ax[7].set_xlabel("abscissa from the inlet $s$ (mm)"), ax[7].set_ylabel("inscribed radius $R$ (mm)")
    ax[7].set_ylim(0, None)
    r8 = ax[7].twinx()
    for r in vres:
        d = r["d"].numpy() / hv
        r8.scatter(r["s_ref"].numpy() * MM, d, c=d, cmap="Reds", vmin=0, vmax=2.5, s=4)
    r8.set_ylim(0, 8), r8.grid(False), r8.set_ylabel("distance to VMTK $d/h$ (voxels)", fontsize=8)
    ax[7].legend(loc="upper right", fontsize=6.5)
    dm = max(r["d"].mean().item() for r in vres) / hv
    ax[7].set_title(f"centreline vs VMTK: mean distance ≤ {dm:.2f} voxel", fontsize=10)
    fig.suptitle(f"Educational voxel CFD — {title} — not for clinical use", fontsize=10)
    n_cyc = len(rec["t"])

    def update(k: int):
        if k < n_cyc:
            j = int(np.abs(wss_t - rec["t"][k]).argmin())
            surf.set_facecolor(cmap(norm1(face(wss[j]))))
            t1.set_text(f"|WSS|, t = {tf[k]:.2f} s")
            s_k, c_k = segs(k)
            stream.set_segments(s_k), stream.set_array(c_k)
            part._offsets3d = tuple(Pp[k].T)
            part.set_color(np.where(in_sac[k], "#dc2626", "#1e3a8a"))
            img.set_data(np.ma.masked_array(sl[k], solid).T)
            if dimg is not None:
                jj = int(np.abs((tref - tf[k] + 0.5) % 1 - 0.5).argmin())  # nearest OpenFOAM sample (periodic)
                dimg.set_array(du[:, jj])
            for c in (cursor, cursor7):
                c.set_xdata([tf[k], tf[k]])
        else:
            surf.set_facecolor(cmap(norm1(face(TAc))))
            t1.set_text("TAWSS (time-averaged |WSS|)")
            part.set_visible(False), stream.set_visible(False), shell.set_visible(False)
            osi_surf.set_visible(True)
            cb_s.update_normal(osi_sm)
            cb_s.set_label("OSI (dimensionless)")
            t2.set_text(f"OSI on the wall (plasma, 0 … {osi_max:.2f}, dimensionless)")

    fig.tight_layout()
    frames = list(range(n_cyc)) + [n_cyc] * N_SUMMARY
    return save_gif_shared_palette(fig, update, frames, path, fps=10)


def _in_sac(P: torch.Tensor, mesh: dict, sac: torch.Tensor) -> np.ndarray:
    """(T, Np) bool: is each tracer in a sac voxel?"""
    g = ((P - mesh["lo"]) / mesh["h"]).round().long()
    shape = torch.tensor(mesh["shape"])
    ok = ((g >= 0) & (g < shape)).all(-1)
    g = g.clamp(min=0).minimum(shape - 1)
    cid = mesh["idx"][g[..., 0], g[..., 1], g[..., 2]]
    return (ok & (cid >= 0) & sac[cid.clamp(min=0)]).numpy()


def _wall_on_sac(grid: dict, mesh: dict, sac: torch.Tensor) -> np.ndarray:
    return (torch.cdist(grid["V"].float(), mesh["pos"][sac].float()).min(1).values < mesh["h"]).numpy()


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 10) -> Path:
    """Render every frame, then quantise all of them with ONE palette built from a strip of frames, so a given
    value keeps the same colour throughout the GIF (Pillow would otherwise pick a new palette per frame)."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)  # numbers behind the final frame (Results page)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'geometry': 'lausanne',
    'voxel': 0.4,
    'cycles': 1.5,
    'dt_ms': 2.0,
    'frames': 24
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.aneurysm_cfd")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.aneurysm_cfd.main")
problem_dir = Path("problems/medical/aneurysm_cfd")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
